<h1 align="center"><b>Ray Tracing in Optical Resonators</b></h1>

<h2>ABCD Matrix Analysis, Cavity Stability, and Ray-Path Visualization</h2>

This notebook studies ray propagation inside two-mirror optical resonators using paraxial ray optics and the ABCD matrix method. It implements spherical mirror cavity geometry, evaluates resonator stability using the $g$ parameters, and visualizes both single-ray and multiple-ray trajectories. Static plots and animations are used to compare stable, marginal, and unstable cavity configurations involving concave and convex mirrors.

## Table of Contents

- [1. Introduction](#1-introduction)
- [2. Theory of Optical Cavity Ray Tracing](#2-theory-of-optical-cavity-ray-tracing)
  - [2.1 Paraxial Ray Vector](#21-paraxial-ray-vector)
  - [2.2 Free-Space Propagation](#22-free-space-propagation)
  - [2.3 Mirror Reflection Matrix](#23-mirror-reflection-matrix)
  - [2.4 Round-Trip Matrix](#24-round-trip-matrix)
  - [2.5 Cavity Stability and the g Parameters](#25-cavity-stability-and-the-g-parameters)
  - [2.6 Stability of Concave and Convex Mirror Cavities](#26-stability-of-concave-and-convex-mirror-cavities)
- [3. Notebook Setup](#3-notebook-setup)
  - [3.1 Imports](#31-imports)
  - [3.2 Plot Style and Output Directories](#32-plot-style-and-output-directories)
  - [3.3 Interactive Matplotlib Backend](#33-interactive-matplotlib-backend)
- [4. Shared Utilities](#4-shared-utilities)
- [5. Cavity Model and Ray Tracing](#5-cavity-model-and-ray-tracing)
  - [5.1 Cavity Parameter Container](#51-cavity-parameter-container)
  - [5.2 CavityRayTracing Class](#52-cavityraytracing-class)
  - [5.3 Ray Propagation Implementation](#53-ray-propagation-implementation)
- [6. Static Plotting Helpers](#6-static-plotting-helpers)
- [7. Animated Ray Tracing](#7-animated-ray-tracing)
  - [7.1 Animation Concept](#71-animation-concept)
  - [7.2 AnimateCavityRayTracing Class](#72-animatecavityraytracing-class)
  - [7.3 Animation Saving](#73-animation-saving)
- [8. Predefined Cavity Configurations](#8-predefined-cavity-configurations)
  - [8.1 Concave-Concave Cavities](#81-concave-concave-cavities)
  - [8.2 Concave-Convex Cavities](#82-concave-convex-cavities)
  - [8.3 Unstable and Marginal Cases](#83-unstable-and-marginal-cases)
- [9. Static Comparison Plots](#9-static-comparison-plots)
- [10. Stability Diagram](#10-stability-diagram)
  - [10.1 Stability Region in the g Plane](#101-stability-region-in-the-g-plane)
  - [10.2 Interpreting the Diagram](#102-interpreting-the-diagram)
  - [10.3 Stability Diagram Implementation](#103-stability-diagram-implementation)
- [11. Symmetric Confocal Optical Cavity](#11-symmetric-confocal-optical-cavity)
  - [11.1 Confocal Stability](#111-confocal-stability)
  - [11.2 Static Confocal Ray Plot](#112-static-confocal-ray-plot)
  - [11.3 Confocal Ray Animation](#113-confocal-ray-animation)
- [12. Multiple-Ray Bundle Tracing](#12-multiple-ray-bundle-tracing)
- [13. Example Studies and Outputs](#13-example-studies-and-outputs)
- [14. References](#14-references)

<a id="1-introduction"></a>
## 1. Introduction

An **optical resonator** (or optical cavity) is a pair of mirrors facing each other so that light bounces back and forth between them. Resonators are the heart of every laser: they store light, define the beam axis, and select the transverse and longitudinal modes of the output.

Before turning to a full Gaussian beam treatment, it is instructive to ask a simpler question: *if a light ray starts slightly off the optical axis, do repeated reflections keep it near the axis, or push it out of the cavity?* Ray tracing answers this question with elementary linear algebra. It does not describe diffraction or the Gaussian field profile, but it gives an intuitive picture of whether a cavity tends to **refocus** or **defocus** small ray deviations — which is exactly what the standard stability criterion quantifies.

**Scope of this notebook**

- Paraxial rays (small heights and angles) in two-mirror cavities with spherical mirrors.
- The ABCD (ray transfer matrix) method for propagation and reflection.
- Cavity stability analysis via the $g_1$, $g_2$ parameters and the stability diagram.
- Static ray-path plots and smooth ray-tracing animations for stable, marginal, and unstable configurations.

**Sign convention used throughout the code**

- **Concave** mirrors use a **negative** radius of curvature, $R < 0$ (e.g. $R_1 = -80$ cm).
- **Convex** mirrors use a **positive** radius of curvature, $R > 0$ (e.g. $R_1 = +80$ cm).

All stability formulas in this notebook are written for this convention, so keep it in mind when comparing with textbooks that use the opposite choice.

<a id="2-theory-of-optical-cavity-ray-tracing"></a>
## 2. Theory of Optical Cavity Ray Tracing

<a id="21-paraxial-ray-vector"></a>
### **2.1 Paraxial Ray Vector**

In paraxial optics a ray at a given plane is fully described by two numbers: its transverse displacement $y$ from the optical axis and its small angle $\theta$ with respect to the axis. We collect them into a state vector

$$
\mathbf{r} =
\begin{pmatrix}
y \\
\theta
\end{pmatrix}
$$

Every optical element (a stretch of free space, a mirror, a lens) acts on this vector as a $2\times 2$ matrix multiplication. A full trip through the cavity is then just a product of matrices applied to the initial ray vector.

<a id="22-free-space-propagation"></a>
### **2.2 Free-Space Propagation**

Propagation over a distance $d$ is described by

$$
P(d) =
\begin{pmatrix}
1 & d \\
0 & 1
\end{pmatrix}
$$

so that

$$
\begin{pmatrix}
y_2 \\
\theta_2
\end{pmatrix} =
\begin{pmatrix}
1 & d \\
0 & 1
\end{pmatrix}
\begin{pmatrix}
y_1 \\
\theta_1
\end{pmatrix}
$$

Intuitively: while the ray travels, its **height changes according to its angle** ($y_2 = y_1 + d\,\theta_1$), but the **angle itself stays constant** in free space ($\theta_2 = \theta_1$).

<a id="23-mirror-reflection-matrix"></a>
### **2.3 Mirror Reflection Matrix**

A spherical mirror leaves the ray height unchanged at the moment of reflection but changes the ray **angle** by an amount proportional to the ray height and the mirror curvature. Consistent with the sign convention of this notebook (concave $R < 0$, convex $R > 0$), the implementation uses the mirror matrix

$$
M(R) =
\begin{pmatrix}
1 & 0 \\
\dfrac{2}{R} & 1
\end{pmatrix}
$$

- For a **concave** mirror, $R < 0$, so the $2/R$ element is **negative**: a ray high above the axis is bent back **toward** the axis. The mirror focuses.
- For a **convex** mirror, $R > 0$, so the $2/R$ element is **positive**: deviations are amplified and the mirror defocuses.

Note that many textbooks write the mirror matrix as $\begin{pmatrix} 1 & 0 \\ -2/R & 1 \end{pmatrix}$ with $R > 0$ for concave mirrors — that is the same physics with the opposite sign convention. The code in this notebook consistently uses $+2/R$ together with $R<0$ for concave mirrors.

<a id="24-round-trip-matrix"></a>
### **2.4 Round-Trip Matrix**

One full round trip in a two-mirror cavity of length $L$ consists of: propagate $L$, reflect off mirror 2, propagate $L$ back, reflect off mirror 1. The round-trip matrix is the ordered product

$$
M_{\mathrm{rt}} = M(R_1)\, P(L)\, M(R_2)\, P(L)
$$

and the ray state after $n+1$ round trips follows from the state after $n$ round trips by

$$
\mathbf{r}_{n+1} = M_{\mathrm{rt}}\,\mathbf{r}_{n}
$$

Whether repeated application of $M_{\mathrm{rt}}$ keeps $\mathbf{r}_n$ **bounded** (the ray stays near the axis forever) or lets it **grow** (the ray walks off the mirrors) is precisely the question of cavity stability.

<a id="25-cavity-stability-and-the-g-parameters"></a>
### **2.5 Cavity Stability and the g Parameters**

With the sign convention of this notebook (concave $R<0$), define the cavity $g$ parameters as

$$
g_1 = 1 + \frac{L}{R_1}, \qquad
g_2 = 1 + \frac{L}{R_2}
$$

The eigenvalues of the round-trip matrix stay on the unit circle — i.e. paraxial rays remain bounded — exactly when

$$
0 < g_1 g_2 < 1
$$

The boundary cases

$$
g_1g_2 = 0 \quad \text{or} \quad g_1g_2 = 1
$$

are **marginal**: the ray motion neither converges nor diverges robustly, and infinitesimal perturbations of the geometry can tip the cavity either way. (The code treats the closed interval $0 \le g_1g_2 \le 1$ as "stable", so marginal configurations are reported as stable; the text distinguishes them explicitly where relevant.)

Intuition:

- If $g_1 g_2$ lies between 0 and 1, the cavity repeatedly refocuses small ray deviations.
- If $g_1 g_2 > 1$, deviations grow monotonically and the cavity is unstable.
- If $g_1 g_2 < 0$, the ray transformation is also unstable for a two-mirror resonator.

<a id="26-stability-of-concave-and-convex-mirror-cavities"></a>
### **2.6 Stability of Concave and Convex Mirror Cavities**

The mirror combination determines which part of the stability range is accessible:

- **Concave–concave** cavities can be stable for suitable values of $L$, $R_1$, and $R_2$, because both mirrors can provide refocusing. Classic examples (confocal, near-planar, near-concentric) all live in this family.
- **Concave–convex** cavities may be stable only in certain parameter ranges: one mirror focuses while the other defocuses, and stability requires the focusing action to win on average over a round trip.
- **Convex–convex** cavities can never form a stable two-mirror resonator under this sign convention. Since convex mirrors have $R > 0$, for any positive cavity length $L$

$$
g_1 = 1 + \frac{L}{R_1} > 1, \qquad
g_2 = 1 + \frac{L}{R_2} > 1
$$

so that

$$
g_1 g_2 > 1
$$

which violates the stability condition — every ray bundle diverges out of the cavity.

<a id="3-notebook-setup"></a>
## 3. Notebook Setup

This section contains only setup code: imports, the Matplotlib style used by all figures, and the output directories where plots and animations are saved.

<a id="31-imports"></a>
### **3.1 Imports**

In [ ]:
import datetime
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Arc, FancyArrowPatch

<a id="32-plot-style-and-output-directories"></a>
### **3.2 Plot Style and Output Directories**

All figures share a dark theme configured once through `plt.rcParams`. Static plots are saved under `OUTPUTS/PLOTS` and animations under `OUTPUTS/ANIMATIONS`. A session timestamp is used to build unique default filenames.

In [ ]:
# Configure standard Matplotlib styling
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "figure.titlesize": 18,
        "figure.titleweight": "bold",
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "axes.labelweight": "bold",
        "legend.fontsize": 11,
        "figure.dpi": 100,
        "figure.facecolor": "black",
        "axes.facecolor": "#00001A",
        "text.color": "white",
        "axes.labelcolor": "white",
        "axes.titlecolor": "white",
        "axes.edgecolor": "white",
    }
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIR = Path("OUTPUTS")
PLOT_DIR = OUTPUT_DIR / "PLOTS"
ANIMATION_DIR = OUTPUT_DIR / "ANIMATIONS"

for folder in [OUTPUT_DIR, PLOT_DIR, ANIMATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print(f"Output directories established under: {OUTPUT_DIR.resolve()}")

<a id="33-interactive-matplotlib-backend"></a>
### **3.3 Interactive Matplotlib Backend**

The `ipympl` backend enables interactive figures and in-notebook animations. It is kept in its own cell because it is notebook magic, not standard Python.

In [ ]:
%matplotlib ipympl

<a id="4-shared-utilities"></a>
## 4. Shared Utilities

Ray tracing produces a list of straight **segments** (mirror-to-mirror chords). For smooth animation, each segment is subdivided into many intermediate points so the ray front can advance gradually.

Interpolation is a *visualization* concern rather than core cavity physics, so it lives here as a single shared helper rather than inside the ray-tracing classes. It is used by `AnimateCavityRayTracing.animate_cavity`, `animate_confocal_ray_tracing`, and `multiple_ray_trace_cavity`, replacing what were previously duplicate implementations.

In [ ]:
def interpolate_segments(segments, points_per_segment=50):
    """
    Interpolate ray segments into evenly spaced points for smooth animation.

    Parameters
    ----------
    segments : list
        List of ray path segments, each a list of (x, y) tuples
    points_per_segment : int
        Number of interpolation points per segment (minimum 2)

    Returns
    -------
    all_points : list
        List of (x, y) coordinates for all interpolated points
    segment_indices : list
        Segment index for each point (for color mapping / progress tracking)
    point_indices_in_segment : list
        Index of each point within its segment
    npps : int
        Actual number of points per segment used
    """
    all_points = []
    segment_indices = []
    point_indices_in_segment = []
    npps = int(max(2, points_per_segment))

    for s_idx, seg in enumerate(segments):
        if len(seg) < 2:
            continue
        (x0, y0), (x1, y1) = seg[0], seg[1]
        t_values = np.linspace(0.0, 1.0, npps)
        for j, t in enumerate(t_values):
            all_points.append((x0 + t * (x1 - x0), y0 + t * (y1 - y0)))
            segment_indices.append(s_idx)
            point_indices_in_segment.append(j)

    return all_points, segment_indices, point_indices_in_segment, npps

<a id="5-cavity-model-and-ray-tracing"></a>
## 5. Cavity Model and Ray Tracing

This is the core implementation section. The cavity model owns parameter validation, geometry, stability analysis, mirror-arc generation, and single-ray tracing.

<a id="51-cavity-parameter-container"></a>
### **5.1 Cavity Parameter Container**

`CavityParameters` is a plain data container holding the cavity geometry ($R_1$, $R_2$, $L$), the derived stability quantities ($g_1$, $g_2$, $g_1 g_2$), and descriptive flags. It is implemented as a `@dataclass`, which keeps the same attribute-based interface as the original class with far less boilerplate.

In [ ]:
@dataclass
class CavityParameters:
    """Container for cavity configuration and stability parameters."""

    R1: float
    R2: float
    L: float
    g1: float
    g2: float
    stability_product: float
    is_stable: bool
    is_confocal: bool
    is_symmetric: bool
    mirror1_type: str
    mirror2_type: str
    cavity_config: str
    mirror_description: str


<a id="52-cavityraytracing-class"></a>
### **5.2 The `CavityRayTracing` Class**

`CavityRayTracing` is the parent class and single owner of:

- **Parameter validation** — rejects zero radii, non-positive lengths, and radii too small for the cavity.
- **Cavity geometry** — mirror centers, centers of curvature, and focal points.
- **Stability calculations** — $g_1$, $g_2$, the stability product, and configuration flags.
- **Mirror arc generation** — `_get_mirror_arcs` is the *only* place mirror-arc geometry is computed; every plotting function reuses it. It is geometry-aware: it depends on $R_1$, $R_2$, $L$, mirror orientation, and the radius sign convention.
- **Single-ray tracing** — `trace_ray` (segments) and `trace_single_ray` (continuous path).
- **Basic static visualization** — `visualize_single_ray`, plus an interactive CLI mode.

In [ ]:
class CavityRayTracing:
    """
    Parent class for ray tracing in optical cavities with concave and/or convex mirrors.

    This base class provides core functionality for general cavity ray tracing, including:
    - Mirror geometry and intersection calculations for both concave (R < 0) and convex (R > 0) mirrors
    - Ray transfer matrix operations (ABCD matrix method)
    - Stability parameter calculation and analysis
    - Parameter validation and error handling
    - Basic static visualization and figure export utilities

    This class supports all cavity configurations: Concave-Concave, Convex-Convex, and Concave-Convex.

    Key Features:
    - Single ray path visualization with dynamic titles and annotations
    - Proper sign handling for concave (negative R) vs convex (positive R) mirrors
    - Stability parameter display (g1, g2) and product checks
    - Interactive CLI parameter input
    """

    def __init__(self, R1=-80.0, R2=-80.0, L=70.0):
        """
        Initialize the cavity ray tracer.

        Parameters
        ----------
        R1 : float
            Radius of curvature for mirror 1 (negative for concave, positive for convex)
            Default: -80.0 cm
        R2 : float
            Radius of curvature for mirror 2 (negative for concave, positive for convex)
            Default: -80.0 cm
        L : float
            Cavity length (distance between mirror vertices)
            Default: 70.0 cm

        Notes
        -----
        - For concave mirrors, use negative R values (e.g., R1 = -80.0)
        - For convex mirrors, use positive R values (e.g., R1 = 80.0)
        - Stability requires: 0 <= g1*g2 <= 1, where g = 1 + L/R
        - Confocal condition: L = |R1| = |R2| (only for concave-concave cavities)

        Raises
        ------
        ValueError
            If parameters are invalid
        """
        self._validate_cavity_parameters(R1, R2, L)
        self.R1 = R1
        self.R2 = R2
        self.L = L
        self._calculate_cavity_properties()

        # Theme color mapping based on ray color
        self.color_dict = {
            "red": {
                "ax_face": "#110000",
                "timer_bg": "#470000",
                "params_bg": "#FFB8B8",
            },
            "blue": {
                "ax_face": "#00001A",
                "timer_bg": "#000047",
                "params_bg": "#B8B8FF",
            },
            "green": {
                "ax_face": "#001100",
                "timer_bg": "#004700",
                "params_bg": "#B8FFB8",
            },
            "orange": {
                "ax_face": "#090500",
                "timer_bg": "#471F00",
                "params_bg": "#FFD8B8",
            },
            "purple": {
                "ax_face": "#0C000C",
                "timer_bg": "#470047",
                "params_bg": "#FFB8FF",
            },
        }

    def _validate_cavity_parameters(self, R1, R2, L):
        """
        Validate cavity parameters.

        Parameters
        ----------
        R1, R2 : float
            Radii of curvature
        L : float
            Cavity length

        Raises
        ------
        ValueError
            If any parameter is invalid
        TypeError
            If parameters are not numeric
        """
        # Type checking
        if not all(isinstance(x, (int, float)) for x in [R1, R2, L]):
            raise TypeError("R1, R2, and L must be numeric values")

        # Value validation
        if R1 == 0:
            raise ValueError("R1 cannot be zero")
        if R2 == 0:
            raise ValueError("R2 cannot be zero")
        if L <= 0:
            raise ValueError("Cavity length L must be positive")

        # Physical constraints (check against absolute radius)
        if abs(R1) < L / 10:
            raise ValueError("R1 appears too small for the cavity length")
        if abs(R2) < L / 10:
            raise ValueError("R2 appears too small for the cavity length")

    def _calculate_cavity_properties(self):
        """Calculate and store cavity properties (g-parameters, stability, etc.)."""
        self.g1 = 1 + self.L / self.R1
        self.g2 = 1 + self.L / self.R2
        self.stability_product = self.g1 * self.g2
        self.is_stable = 0 <= self.stability_product <= 1

        # Determine mirror types
        self.mirror1_type = "Concave" if self.R1 < 0 else "Convex"
        self.mirror2_type = "Concave" if self.R2 < 0 else "Convex"

        # Determine cavity configuration string
        if self.R1 < 0 and self.R2 < 0:
            self.cavity_config = "Concave-Concave"
        elif self.R1 > 0 and self.R2 > 0:
            self.cavity_config = "Convex-Convex"
        elif self.R1 < 0 and self.R2 > 0:
            self.cavity_config = "Concave-Convex"
        else:
            self.cavity_config = "Convex-Concave"

        # Determine mirror description string for titles
        if self.mirror1_type == self.mirror2_type:
            self.mirror_description = f"{self.mirror1_type}"
        else:
            self.mirror_description = f"{self.mirror1_type}-{self.mirror2_type}"

        # Check if confocal (only applies to concave-concave with L = |R1| = |R2|)
        self.is_confocal = (
            self.R1 < 0
            and self.R2 < 0
            and abs(self.L - abs(self.R1)) < 1e-6
            and abs(abs(self.R1) - abs(self.R2)) < 1e-6
        )

        # Check if Symmetric (radii are identical, including sign)
        self.is_symmetric = abs(self.R1 - self.R2) < 1e-6

        # Mirror centers for plotting
        self.left_mirror_center_x = -self.L / 2 - self.R1
        self.right_mirror_center_x = self.L / 2 + self.R2

        # focal points
        x_pole_L = -self.L / 2.0
        x_pole_R = +self.L / 2.0
        self.F1x = x_pole_L - self.R1 * 0.5
        self.F2x = x_pole_R + self.R2 * 0.5

        # centers of curvature
        self.C1x = self.left_mirror_center_x
        self.C2x = self.right_mirror_center_x

    def get_cavity_parameters(self):
        """
        Get cavity parameters and stability information.

        Returns
        -------
        CavityParameters
            Object containing all cavity parameters
        """
        return CavityParameters(
            R1=self.R1,
            R2=self.R2,
            L=self.L,
            g1=self.g1,
            g2=self.g2,
            stability_product=self.stability_product,
            is_stable=self.is_stable,
            is_confocal=self.is_confocal,
            is_symmetric=self.is_symmetric,
            mirror1_type=self.mirror1_type,
            mirror2_type=self.mirror2_type,
            cavity_config=self.cavity_config,
            mirror_description=self.mirror_description,
        )

    def _get_x_on_mirror(self, y, R, center_x, side):
        """
        Calculate the x-coordinate on a spherical mirror for a given y.

        Parameters
        ----------
        y : float
            Y-coordinate
        R : float
            Radius of curvature
        center_x : float
            X-coordinate of mirror center
        side : str
            'left' or 'right' indicating which mirror

        Returns
        -------
        float
            X-coordinate on mirror surface, or np.nan if invalid
        """
        if abs(y) > abs(R):
            return np.nan

        discriminant = R**2 - y**2
        if discriminant < 0:
            return np.nan

        if side == "left":
            return center_x + np.sign(R) * np.sqrt(discriminant)
        elif side == "right":
            return center_x - np.sign(R) * np.sqrt(discriminant)
        else:
            return np.nan

    def _get_mirror_arcs(self, arc_angle):
        """Helper to create left and right mirror Arc patches properly oriented for concave or convex mirrors."""
        if self.R1 < 0:
            theta1_left, theta2_left = 180 - arc_angle, 180 + arc_angle
        else:
            theta1_left, theta2_left = -arc_angle, arc_angle

        if self.R2 < 0:
            theta1_right, theta2_right = -arc_angle, arc_angle
        else:
            theta1_right, theta2_right = 180 - arc_angle, 180 + arc_angle

        left_mirror = Arc(
            (self.left_mirror_center_x, 0),
            2 * abs(self.R1),
            2 * abs(self.R1),
            angle=0,
            theta1=theta1_left,
            theta2=theta2_left,
            color="gray",
            lw=6,
            zorder=5,
        )
        right_mirror = Arc(
            (self.right_mirror_center_x, 0),
            2 * abs(self.R2),
            2 * abs(self.R2),
            angle=0,
            theta1=theta1_right,
            theta2=theta2_right,
            color="gray",
            lw=6,
            zorder=5,
        )
        return left_mirror, right_mirror

    def _validate_ray_parameters(self, y0_initial, theta0_initial_deg, N_round_trips):
        """
        Validate ray tracing parameters.

        Parameters
        ----------
        y0_initial : float
            Initial ray height
        theta0_initial_deg : float
            Initial angle in degrees
        N_round_trips : int
            Number of round trips

        Raises
        ------
        ValueError
            If parameters are invalid
        TypeError
            If parameters have wrong type
        """
        if not isinstance(N_round_trips, int):
            raise TypeError("N_round_trips must be an integer")

        if N_round_trips < 1:
            raise ValueError("N_round_trips must be at least 1")

        if N_round_trips > 100:
            raise ValueError("N_round_trips too large (max 100)")

        if not isinstance(y0_initial, (int, float)):
            raise TypeError("y0_initial must be numeric")

        if not isinstance(theta0_initial_deg, (int, float)):
            raise TypeError("theta0_initial_deg must be numeric")

        if abs(y0_initial) > abs(self.R1) * 0.9:
            raise ValueError(f"Initial height too large (max {abs(self.R1) * 0.9:.1f})")

        if abs(theta0_initial_deg) > 45:
            raise ValueError("Initial angle too large (max ±45°)")

    def trace_ray(self, y0_initial=15.0, theta0_initial_deg=0.0, N_round_trips=2):
        """
        Perform ray tracing through the cavity.

        Parameters
        ----------
        y0_initial : float
            Initial height of ray from optical axis
        theta0_initial_deg : float
            Initial angle in degrees
        N_round_trips : int
            Number of round trips to simulate

        Returns
        -------
        ray_segments : list
            List of ray path segments, each segment is a list of (x, y) tuples
        final_state : numpy.ndarray
            Final state vector [y, theta] after all round trips

        Raises
        ------
        ValueError
            If ray parameters are invalid
        """
        self._validate_ray_parameters(y0_initial, theta0_initial_deg, N_round_trips)

        # Degrees to radians
        theta0_initial = np.deg2rad(theta0_initial_deg)

        # Create transfer matrices
        M_prop = np.array([[1, self.L], [0, 1]])
        M_refl_1 = np.array([[1, 0], [2 / self.R1, 1]])
        M_refl_2 = np.array([[1, 0], [2 / self.R2, 1]])

        # Initialize ray tracing
        ray_segments = []
        current_segment = []

        # Initial state vector [y, theta]
        y_theta_vec = np.array([[y0_initial], [theta0_initial]])

        # Initial point on Mirror 1
        x0 = self._get_x_on_mirror(
            y0_initial, self.R1, self.left_mirror_center_x, "left"
        )

        if np.isnan(x0):
            raise ValueError("Initial ray position is outside mirror aperture")

        current_segment.append((x0, y0_initial))

        # Ray tracing
        for i in range(N_round_trips):
            # Propagation from Mirror 1 to Mirror 2
            y_theta_vec = M_prop @ y_theta_vec
            y_at_M2 = y_theta_vec[0, 0]
            x_at_M2 = self._get_x_on_mirror(
                y_at_M2, self.R2, self.right_mirror_center_x, "right"
            )

            if np.isnan(x_at_M2):
                print(
                    f"\n[Notice] Ray escaped at Mirror 2 during round trip {i + 1} "
                    f"(height |y| = {abs(y_at_M2):.2f} cm > aperture |R2| = {abs(self.R2):.2f} cm)."
                )
                current_segment.append((self.L / 2, y_at_M2))
                ray_segments.append(current_segment.copy())
                break

            current_segment.append((x_at_M2, y_at_M2))
            ray_segments.append(current_segment.copy())
            current_segment = [(x_at_M2, y_at_M2)]

            # Reflection from Mirror 2
            y_theta_vec = M_refl_2 @ y_theta_vec

            # Propagation from Mirror 2 back to Mirror 1
            y_theta_vec = M_prop @ y_theta_vec
            y_at_M1 = y_theta_vec[0, 0]
            x_at_M1 = self._get_x_on_mirror(
                y_at_M1, self.R1, self.left_mirror_center_x, "left"
            )

            if np.isnan(x_at_M1):
                print(
                    f"\n[Notice] Ray escaped at Mirror 1 during round trip {i + 1} "
                    f"(height |y| = {abs(y_at_M1):.2f} cm > aperture |R1| = {abs(self.R1):.2f} cm)."
                )
                current_segment.append((-self.L / 2, y_at_M1))
                ray_segments.append(current_segment.copy())
                break

            current_segment.append((x_at_M1, y_at_M1))
            ray_segments.append(current_segment.copy())
            current_segment = [(x_at_M1, y_at_M1)]

            # Reflection from Mirror 1
            y_theta_vec = M_refl_1 @ y_theta_vec

        return ray_segments, y_theta_vec

    def print_info(self, y0_initial, theta0_initial_deg, N_round_trips):
        """Print detailed cavity information and simulation parameters."""
        params = self.get_cavity_parameters()

        print("=" * 50)
        print("CAVITY INFORMATION")
        print("=" * 50)
        print(
            f"Mirror 1 radius (R1):        {params.R1:.2f} cm ({params.mirror1_type})"
        )
        print(
            f"Mirror 2 radius (R2):        {params.R2:.2f} cm ({params.mirror2_type})"
        )
        print(f"Cavity length (L):           {params.L:.2f} cm")
        print("-" * 50)
        print(f"Stability parameter g1:      {params.g1:.4f}")
        print(f"Stability parameter g2:      {params.g2:.4f}")
        print(f"Stability product (g1*g2):   {params.stability_product:.4f}")
        print("-" * 50)

        cavity_symmetry = "Symmetric" if params.is_symmetric else "Asymmetric"
        if params.is_confocal:
            cavity_type = "CONFOCAL"
        elif params.is_symmetric:
            cavity_type = f"Symmetric {params.mirror_description}"
        else:
            cavity_type = f"Asymmetric {params.mirror_description}"
        print(
            f"Cavity configuration:        {params.cavity_config} ({cavity_symmetry})"
        )
        print(f"Cavity type:                 {cavity_type}")
        print(
            f"Stability status:            {'STABLE' if params.is_stable else 'UNSTABLE'}"
        )
        if params.is_stable:
            print("[+] Cavity is stable for beam propagation")
        else:
            print("[-] Cavity is unstable - beam will diverge")
        print("=" * 50)
        print("SIMULATION PARAMETERS")
        print("=" * 50)
        print(f"Initial ray height (y0):     {y0_initial:.2f} cm")
        print(f"Initial angle (theta0):      {theta0_initial_deg:.2f} deg")
        print(f"Number of round trips:       {N_round_trips}")
        print("=" * 50)

    def trace_single_ray(
        self, y0_initial=15.0, theta0_initial_deg=0.0, N_round_trips=50
    ):
        """
        Trace a single ray through the cavity.

        This method uses the parent class's trace_ray method but formats
        the output as a single continuous path rather than segments.

        Parameters
        ----------
        y0_initial : float
            Initial height of ray from optical axis (cm)
        theta0_initial_deg : float
            Initial angle in degrees
        N_round_trips : int
            Number of round trips to simulate

        Returns
        -------
        ray_path : list of tuples
            List of (x, y) coordinates along the ray path
        final_state : numpy.ndarray
            Final state vector [y, theta] after all round trips

        Raises
        ------
        ValueError
            If ray escapes the cavity or parameters are invalid
        """
        ray_segments, final_state = self.trace_ray(
            y0_initial, theta0_initial_deg, N_round_trips
        )

        ray_path = []
        for i, segment in enumerate(ray_segments):
            if i == 0:
                ray_path.extend(segment)
            else:
                ray_path.extend(segment[1:])

        return ray_path, final_state

    def visualize_single_ray(
        self,
        y0_initial=15.0,
        theta0_initial_deg=0.0,
        N_round_trips=50,
        arc_angle=25,
        ray_color="red",
        save_figure=False,
        filename=None,
    ):
        """
        Visualize single ray tracing through the optical cavity.

        Creates a detailed plot showing:
        - Mirror geometry with arcs (oriented for concave or convex)
        - Optical axis
        - Ray path with markers
        - Design parameters annotation
        - Cavity stability status

        Parameters
        ----------
        y0_initial : float
            Initial height of ray from optical axis
        theta0_initial_deg : float
            Initial angle in degrees
        N_round_trips : int
            Number of round trips to simulate
        arc_angle : float
            Angle for arc representation of mirrors (0-90 degrees)
        ray_color : str
            Color of the ray ('red', 'blue', 'green', 'orange', 'purple')
        save_figure : bool
            Whether to save the figure as PNG
        filename : str or None
            Filename for saved figure (auto-generated if None)

        Returns
        -------
        fig : matplotlib.figure.Figure
            The figure object
        ax : matplotlib.axes.Axes
            The axes object

        Raises
        ------
        ValueError
            If parameters are invalid or ray escapes cavity
        """
        # arc_angle validation
        if not 0 < arc_angle < 90:
            raise ValueError("arc_angle must be between 0 and 90 degrees")

        # Ray Tracing
        try:
            ray_path, final_state = self.trace_single_ray(
                y0_initial, theta0_initial_deg, N_round_trips
            )
        except Exception as e:
            raise ValueError(f"Ray tracing failed: {str(e)}")

        fig, ax = plt.subplots(figsize=(12, 8))
        fig.subplots_adjust(left=0.02, right=0.8, top=0.94, bottom=0.02)
        ax.set_facecolor(self.color_dict[ray_color]["ax_face"])

        # Mirror arcs
        left_mirror, right_mirror = self._get_mirror_arcs(arc_angle)
        ax.add_patch(left_mirror)
        ax.add_patch(right_mirror)

        # Optical axis
        ax.plot(
            [-self.L / 2, self.L / 2], [0, 0], color="black", lw=1, ls="--", zorder=1
        )

        # Plotting ray path
        path_x, path_y = zip(*ray_path)
        ax.plot(
            path_x,
            path_y,
            "-o",
            color=ray_color,
            markersize=3,
            label="Ray Path",
            zorder=3,
        )

        # Add design parameters annotation
        param_text = (
            f"Design Parameters\n"
            f"{'-' * 23}\n"
            f"$R_1$ = {self.R1:.1f} cm ({self.mirror1_type})\n"
            f"$R_2$ = {self.R2:.1f} cm ({self.mirror2_type})\n"
            f"$L$ = {self.L:.1f} cm\n"
            f"$g_1$ = {self.g1:.3f}\n"
            f"$g_2$ = {self.g2:.3f}\n"
            f"$g_1 \\times g_2$ = {self.stability_product:.3f}\n"
            f"Stability: {'STABLE' if self.is_stable else 'UNSTABLE'}\n"
            f"Initial $\\theta$ = {theta0_initial_deg:.1f}°\n"
            f"Round trips: {N_round_trips}"
        )

        ax.text(
            1.02,
            0.98,
            param_text,
            transform=ax.transAxes,
            va="top",
            bbox=dict(
                boxstyle="round",
                facecolor=self.color_dict[ray_color]["params_bg"],
                alpha=0.4,
            ),
            fontsize=10,
            fontfamily="monospace",
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

        if self.is_confocal:
            title_str = "Confocal"
        elif self.is_symmetric:
            title_str = f"Symmetric {self.mirror_description}"
        else:
            title_str = f"Asymmetric {self.mirror_description}"
        ax.set_title(f"Single Ray Tracing in {title_str} Cavity", pad=10)
        ax.legend(loc="upper center")

        if save_figure:
            self._save_figure(fig, filename, theta0_initial_deg, ray_color)

        plt.show()

        return fig, ax

    def _save_figure(self, fig, filename, theta0_initial_deg, ray_color):
        """
        Save figure to file.

        Parameters
        ----------
        fig : matplotlib.figure.Figure
            Figure to save
        filename : str or None
            Filename (auto-generated if None)
        theta0_initial_deg : float
            Initial angle (for auto-generated filename)
        ray_color : str
            Color of the ray (for auto-generated filename metadata)
        """
        if filename is None:
            config_prefix = self.cavity_config.lower().replace("-", "_")
            sym_prefix = "symmetric" if self.is_symmetric else "asymmetric"
            metadata = f"R1_{self.R1}_R2_{self.R2}_L_{self.L}_theta_{theta0_initial_deg}_rc_{ray_color}"
            filename = f"{sym_prefix}_{config_prefix}_cavity_{metadata}_{timestamp}.png"
        try:
            filepath = PLOT_DIR / filename
            fig.savefig(filepath, dpi=300, bbox_inches="tight")
            print(f"Figure saved successfully to: {filepath}")
        except Exception as e:
            print(f"Error saving figure: {e}")

    def run_interactive(self):
        """
        Run interactive single ray tracing with user inputs.

        Provides a command-line interface for users to:
        - Set cavity parameters (R1, R2, L)
        - Configure ray initial conditions
        - Choose visualization options
        - Save figures

        Returns
        -------
        fig : matplotlib.figure.Figure
            The figure object
        ax : matplotlib.axes.Axes
            The axes object
        """
        print("\n" + "=" * 60)
        print("CAVITY SINGLE RAY TRACING - INTERACTIVE MODE")
        print("=" * 60)

        # Default parameters
        defaults = {
            "R1": -80.0,
            "R2": -80.0,
            "L": 70.0,
            "N_round_trips": 50,
            "y0_initial": 15.0,
            "theta0_initial_deg": 0.0,
            "arc_angle": 25,
            "ray_color": "red",
        }

        use_defaults = input("\nUse default parameters? (y/n): ").strip().lower() == "y"

        if use_defaults:
            params = defaults.copy()
            if (
                self.R1 != defaults["R1"]
                or self.R2 != defaults["R2"]
                or self.L != defaults["L"]
            ):
                self.__init__(defaults["R1"], defaults["R2"], defaults["L"])
        else:
            params = defaults.copy()
            try:
                # Get cavity parameters
                R1_input = input(
                    f"Enter radius of curvature for mirror 1 (- for concave, + for convex) [{self.R1}]: "
                ).strip()
                R1 = float(R1_input) if R1_input else self.R1

                R2_input = input(
                    f"Enter radius of curvature for mirror 2 (- for concave, + for convex) [{self.R2}]: "
                ).strip()
                R2 = float(R2_input) if R2_input else self.R2

                L_input = input(f"Enter cavity length [{self.L}]: ").strip()
                L = float(L_input) if L_input else self.L

                if R1 != self.R1 or R2 != self.R2 or L != self.L:
                    self.__init__(R1, R2, L)

                params["R1"] = R1
                params["R2"] = R2
                params["L"] = L

                params["N_round_trips"] = int(
                    input(
                        f"Enter number of round trips [{defaults['N_round_trips']}]: "
                    )
                    or defaults["N_round_trips"]
                )

                params["y0_initial"] = float(
                    input(f"Enter initial ray height [{defaults['y0_initial']}]: ")
                    or defaults["y0_initial"]
                )

                params["theta0_initial_deg"] = float(
                    input(
                        f"Enter initial angle in degrees [{defaults['theta0_initial_deg']}]: "
                    )
                    or defaults["theta0_initial_deg"]
                )

                color_choice = (
                    input("Enter ray color (red/blue/green/orange/purple) [red]: ")
                    .strip()
                    .lower()
                )
                params["ray_color"] = (
                    color_choice
                    if color_choice in ["red", "blue", "green", "orange", "purple"]
                    else defaults["ray_color"]
                )

                params["arc_angle"] = float(
                    input(
                        f"Enter arc angle for mirror display [{defaults['arc_angle']}]: "
                    )
                    or defaults["arc_angle"]
                )

            except ValueError as e:
                print(f"Invalid input: {e}")
                print("Using default parameters.")
                params = defaults.copy()
                if (
                    self.R1 != defaults["R1"]
                    or self.R2 != defaults["R2"]
                    or self.L != defaults["L"]
                ):
                    self.__init__(defaults["R1"], defaults["R2"], defaults["L"])

        # Save figure options
        params["save_figure"] = (
            input("\nSave figure as PNG? (y/n): ").strip().lower() == "y"
        )
        filename = None
        if params["save_figure"]:
            input_filename = input("Enter filename (or press Enter for auto): ").strip()
            if input_filename:
                if not input_filename.endswith(".png"):
                    input_filename += ".png"
                filename = input_filename

        params["filename"] = filename

        # Print cavity info
        self.print_info(
            y0_initial=params["y0_initial"],
            theta0_initial_deg=params["theta0_initial_deg"],
            N_round_trips=params["N_round_trips"],
        )

        params.pop("R1", None)
        params.pop("R2", None)
        params.pop("L", None)

        try:
            fig, ax = self.visualize_single_ray(**params)
            return fig, ax
        except Exception as e:
            print(f"\nError during simulation: {e}")
            raise

<a id="53-ray-propagation-implementation"></a>
### **5.3 Ray Propagation Implementation**

How the theory of Section 2 maps into `trace_ray`:

- **Initial ray.** The user supplies the starting height $y_0$ and angle $\theta_0$ (in degrees, converted to radians). The starting point is placed *on the surface of mirror 1*, using `_get_x_on_mirror` to solve the sphere equation for the $x$-coordinate at height $y_0$.
- **Transfer matrices.** `M_prop` is the free-space matrix $P(L)$ of Section 2.2, and `M_refl_1`, `M_refl_2` are the mirror matrices $M(R_1)$, $M(R_2)$ of Section 2.3 — note the $+2/R$ element, consistent with the concave-$R<0$ convention.
- **Reflection points.** After each propagation the new height is intersected with the actual spherical mirror surface (not a flat plane), so the drawn ray endpoints lie on the mirror arcs. If the height exceeds the mirror aperture, the ray "escapes" and tracing stops with a notice.
- **Repeated round trips.** Each loop iteration applies the sequence propagate → reflect at M2 → propagate → reflect at M1, which is exactly one application of the round-trip matrix $M_{\mathrm{rt}}$.
- **Segment storage.** The path is stored as a list of two-point segments (one chord per mirror-to-mirror transit). Static plots join them into a continuous polyline (`trace_single_ray`), while animations interpolate each segment into many points (Section 4).

<a id="6-static-plotting-helpers"></a>
## 6. Static Plotting Helpers

The standalone plotting and animation functions in Sections 9–12 all need the same canvas: a dark figure with the two mirror arcs, the dashed optical axis, no ticks, and a monospace parameter box. These helpers centralize that setup.

Two rules keep the geometry consistent:

- Mirror arcs are always obtained from `cavity._get_mirror_arcs(arc_angle)` — the helpers never recompute mirror geometry.
- The figure-saving helper is named `export_figure` (not `save_figure`) so it does not collide with the boolean `save_figure` parameters used throughout the plotting functions.

The class methods `visualize_single_ray` and `animate_cavity` predate these helpers and remain self-contained, but they follow the same pattern and the same `_get_mirror_arcs` source of truth.

In [ ]:
def setup_cavity_axes(
    cavity,
    arc_angle=25.0,
    figsize=(12, 8),
    ax_face=None,
    mirror_linewidth=None,
    axis_alpha=0.8,
):
    """
    Create a figure/axes pair with the cavity mirrors and optical axis drawn.

    Mirror arcs always come from cavity._get_mirror_arcs(...) so that the
    mirror geometry has a single source of truth.

    Parameters
    ----------
    cavity : CavityRayTracing
        Cavity instance providing geometry via _get_mirror_arcs.
    arc_angle : float
        Angle for arc representation of mirrors (0-90 degrees).
    figsize : tuple
        Figure size in inches.
    ax_face : str or None
        Axes background color. Defaults to the cavity's blue theme.
    mirror_linewidth : float or None
        Override for the mirror arc line width (None keeps the default).
    axis_alpha : float
        Alpha of the dashed optical axis line.

    Returns
    -------
    fig, ax : matplotlib figure and axes
    """
    fig, ax = plt.subplots(figsize=figsize)

    if ax_face is None:
        ax_face = (
            getattr(cavity, "color_dict", {}).get("blue", {}).get("ax_face", "#e6f3ff")
        )
    ax.set_facecolor(ax_face)

    # Mirrors: single source of truth for arc geometry
    left_mirror, right_mirror = cavity._get_mirror_arcs(arc_angle)
    if mirror_linewidth is not None:
        left_mirror.set_linewidth(mirror_linewidth)
        right_mirror.set_linewidth(mirror_linewidth)
    ax.add_patch(left_mirror)
    ax.add_patch(right_mirror)

    # Optical axis
    ax.plot(
        [-cavity.L / 2, cavity.L / 2],
        [0, 0],
        color="grey",
        lw=1,
        ls="--",
        zorder=1,
        alpha=axis_alpha,
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    return fig, ax


def add_cavity_parameter_box(
    ax,
    cavity=None,
    text=None,
    facecolor="greenyellow",
    alpha=0.4,
    x=1.02,
    y=0.98,
    fontsize=10,
):
    """
    Add a monospace parameter box anchored outside the axes.

    Either pass prebuilt `text`, or pass a `cavity` to get a standard
    parameter summary (R1, R2, L, g1, g2, g1*g2).
    """
    if text is None:
        if cavity is None:
            raise ValueError("Provide either `text` or a `cavity` instance")
        text = (
            f"$R_1$ = {cavity.R1:.1f} cm ({cavity.mirror1_type})\n"
            f"$R_2$ = {cavity.R2:.1f} cm ({cavity.mirror2_type})\n"
            f"$L$ = {cavity.L:.1f} cm\n"
            f"$g_1$ = {cavity.g1:.3f}\n"
            f"$g_2$ = {cavity.g2:.3f}\n"
            f"$g_1 \\times g_2$ = {cavity.stability_product:.3f}"
        )
    return ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor=facecolor, alpha=alpha),
        fontsize=fontsize,
        fontfamily="monospace",
    )


def export_figure(fig, path, dpi=300):
    """Save a figure to `path` with error handling."""
    try:
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Figure saved successfully to: {path}")
    except Exception as e:
        print(f"Error saving figure: {e}")

<a id="7-animated-ray-tracing"></a>
## 7. Animated Ray Tracing

<a id="71-animation-concept"></a>
### **7.1 Animation Concept**

Animation works in two stages. First the ray path is computed as reflection **segments** (Section 5). Then each segment is interpolated into many intermediate points with the shared `interpolate_segments` helper (Section 4), so that the ray front can advance smoothly frame by frame. A moving arrowhead marks the current ray front, and a counter reports completed round trips.

<a id="72-animatecavityraytracing-class"></a>
### **7.2 AnimateCavityRayTracing Class**

`AnimateCavityRayTracing` is a convenience class built on top of `CavityRayTracing`. It inherits all geometry, tracing, and stability functionality, and adds frame-by-frame visualization, GIF export, and an interactive CLI. Its animation method uses the shared `interpolate_segments` helper rather than a private duplicate.

In [ ]:
class AnimateCavityRayTracing(CavityRayTracing):
    """
    Animated ray tracing for optical cavities with concave and/or convex mirrors.

    Inherits from CavityRayTracing to utilize core functionality:
    - Mirror geometry and intersection calculations
    - Ray transfer matrix operations
    - Stability analysis
    - Parameter validation

    This class handles animation-specific features:
    - Frame-by-frame ray propagation visualization
    - Dynamic arrowhead at ray front
    - Real-time round-trip counter
    - GIF export capability

    Correctly handles both concave and convex mirrors with proper orientation and positioning.
    """

    def __init__(self, R1=-80.0, R2=-80.0, L=70.0):
        """
        Initialize the animated cavity ray tracer.

        Parameters
        ----------
        R1 : float
            Radius of curvature for mirror 1 (negative for concave, positive for convex)
            Default: -80.0 cm
        R2 : float
            Radius of curvature for mirror 2 (negative for concave, positive for convex)
            Default: -80.0 cm
        L : float
            Cavity length (distance between mirror vertices)
            Default: 70.0 cm
        """
        # Initialize parent class
        super().__init__(R1, R2, L)

    def animate_cavity(
        self,
        y0_initial=15.0,
        theta0_initial_deg=0.0,
        N_round_trips=50,
        arc_angle=25,
        fps=30,
        points_per_segment=5,
        ray_color="red",
        save_animation=False,
        filename=None,
    ):
        """
        Create animated visualization of ray tracing through the cavity.

        Parameters
        ----------
        y0_initial : float
            Initial ray height from optical axis (cm)
        theta0_initial_deg : float
            Initial angle in degrees
        N_round_trips : int
            Number of round trips to simulate
        arc_angle : float
            Angle for arc representation of mirrors (0-90 degrees)
        fps : int
            Frames per second for animation
        points_per_segment : int
            Number of interpolation points per ray segment
        ray_color : str
            Color of the ray ('red', 'blue', 'green', 'orange', 'purple')
        save_animation : bool
            Whether to save animation as GIF
        filename : str or None
            Filename for saved animation (auto-generated if None)

        Returns
        -------
        fig : matplotlib.figure.Figure
            The figure object
        anim : matplotlib.animation.FuncAnimation
            The animation object
        """
        # Ray tracing and validation using parent class method
        ray_segments, _ = self.trace_ray(y0_initial, theta0_initial_deg, N_round_trips)

        # Interpolate segments for smooth animation
        all_points, seg_ids, pt_idx_in_seg, npps = interpolate_segments(
            ray_segments, points_per_segment
        )
        fig, ax = plt.subplots(figsize=(12, 8))
        fig.subplots_adjust(left=0.02, right=0.8, top=0.94, bottom=0.02)
        ax.set_facecolor(self.color_dict[ray_color]["ax_face"])

        left_mirror, right_mirror = self._get_mirror_arcs(arc_angle)
        ax.add_patch(left_mirror)
        ax.add_patch(right_mirror)

        ax.plot(
            [-self.L / 2, self.L / 2],
            [0, 0],
            color="grey",
            lw=1,
            ls="--",
            zorder=1,
            alpha=0.5,
        )

        # Initialize animated ray line
        (ray_line,) = ax.plot(
            [], [], "-", linewidth=2, color=ray_color, zorder=3, alpha=0.85
        )
        arrow_patch = None

        if self.is_confocal:
            title_str = "Confocal"
        elif self.is_symmetric:
            title_str = f"Symmetric {self.mirror_description}"
        else:
            title_str = f"Asymmetric {self.mirror_description}"

        param_text = (
            f"Design Parameters\n"
            f"{'-' * 23}\n"
            f"$R_1$ = {self.R1:.1f} cm ({self.mirror1_type})\n"
            f"$R_2$ = {self.R2:.1f} cm ({self.mirror2_type})\n"
            f"$L$ = {self.L:.1f} cm\n"
            f"$g_1$ = {self.g1:.3f}\n"
            f"$g_2$ = {self.g2:.3f}\n"
            f"$g_1 \\times g_2$ = {self.stability_product:.3f}\n"
            f"Stability: {'STABLE' if self.is_stable else 'UNSTABLE'}\n"
            f"$\\theta_0$ = {theta0_initial_deg:.1f}°\n"
            f"Round trips = {N_round_trips}"
        )
        ax.text(
            1.02,
            0.98,
            param_text,
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(
                boxstyle="round",
                facecolor=self.color_dict[ray_color]["params_bg"],
                alpha=0.4,
            ),
            fontsize=10,
            fontfamily="monospace",
        )

        round_text = ax.text(
            0.5,
            0.98,
            f"Round trips: 0/{N_round_trips}",
            transform=ax.transAxes,
            va="top",
            ha="center",
            bbox=dict(
                boxstyle="round",
                facecolor=self.color_dict[ray_color]["timer_bg"],
                edgecolor="grey",
                alpha=0.7,
            ),
            fontsize=12,
            fontfamily="monospace",
        )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_title(f"Ray Tracing Animation in {title_str} Cavity", pad=10)

        def animate(frame):
            nonlocal arrow_patch
            if frame >= len(all_points):
                return (ray_line, round_text)

            trail = all_points[: frame + 1]
            trail_x = [p[0] for p in trail]
            trail_y = [p[1] for p in trail]
            ray_line.set_data(trail_x, trail_y)

            s_idx = seg_ids[frame]
            completed = s_idx // 2
            if (s_idx % 2 == 1) and (pt_idx_in_seg[frame] == npps - 1):
                completed += 1
            round_text.set_text(f"Round trips: {completed}/{N_round_trips}")

            if arrow_patch is not None:
                arrow_patch.remove()
                arrow_patch = None

            if frame > 0:
                x_prev, y_prev = all_points[frame - 1]
                x_cur, y_cur = all_points[frame]
                dx, dy = x_cur - x_prev, y_cur - y_prev
                if dx * dx + dy * dy > 1e-9:
                    start_x = x_cur - 0.25 * dx
                    start_y = y_cur - 0.25 * dy
                    arrow_patch = FancyArrowPatch(
                        (start_x, start_y),
                        (x_cur, y_cur),
                        arrowstyle="-|>",
                        mutation_scale=16,
                        color=ray_color,
                        linewidth=2,
                        zorder=6,
                        alpha=0.95,
                    )
                    ax.add_patch(arrow_patch)

            return (ray_line, round_text)

        anim = FuncAnimation(
            fig,
            animate,
            frames=len(all_points),
            interval=int(1000 / fps),
            blit=False,
            repeat=False,
        )

        if save_animation:
            self._save_animation(anim, filename, fps, ray_color)
            plt.close(fig)
        else:
            plt.show()

        return fig, anim

    def _save_animation(self, anim, filename, fps, ray_color):
        """
        Save animation to GIF file.

        Parameters
        ----------
        anim : matplotlib.animation.FuncAnimation
            Animation object to save
        filename : str or None
            Filename for saved animation
        fps : int
            Frames per second
        ray_color : str
            Color of the ray (for auto-generated filename metadata)
        """
        if filename is None:
            config_prefix = self.cavity_config.lower().replace("-", "_")
            sym_prefix = "symmetric" if self.is_symmetric else "asymmetric"
            metadata = f"R1_{self.R1}_R2_{self.R2}_L_{self.L}_rc_{ray_color}"
            filename = f"{sym_prefix}_{config_prefix}_cavity_{metadata}_{timestamp}.gif"
        try:
            filepath = ANIMATION_DIR / filename
            print(f"Saving animation to: {filepath}")
            anim.save(filepath, writer="ffmpeg", fps=fps, dpi=120)
            print("Animation saved successfully!")
        except Exception as e:
            print(f"Error saving animation: {e}")

    def run_interactive(self):
        """
        Interactive CLI for animated ray tracing.

        Prompts user for cavity parameters and animation settings,
        then generates and displays/saves the animation.
        """
        print("=" * 60)
        print("Animated Optical Cavity Ray Tracing")
        print("=" * 60)

        defaults = {
            "y0_initial": 15.0,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 25,
            "arc_angle": 25,
            "fps": 30,
            "points_per_segment": 5,
            "ray_color": "red",
            "save_animation": False,
        }

        use_defaults = (
            input("Use default animation parameters? (y/n): ").strip().lower() == "y"
        )

        if use_defaults:
            params = defaults.copy()
        else:
            params = {}
            try:
                params["y0_initial"] = float(
                    input(f"Enter initial height [{defaults['y0_initial']}]: ")
                    or defaults["y0_initial"]
                )
                params["theta0_initial_deg"] = float(
                    input(
                        f"Enter initial angle (deg) [{defaults['theta0_initial_deg']}]: "
                    )
                    or defaults["theta0_initial_deg"]
                )
                params["N_round_trips"] = int(
                    input(
                        f"Enter number of round trips [{defaults['N_round_trips']}]: "
                    )
                    or defaults["N_round_trips"]
                )
                params["points_per_segment"] = int(
                    input(
                        f"Enter points per segment [{defaults['points_per_segment']}]: "
                    )
                    or defaults["points_per_segment"]
                )
                color_choice = (
                    input("Enter ray color (red/blue/green/orange/purple) [red]: ")
                    .strip()
                    .lower()
                )
                params["ray_color"] = (
                    color_choice
                    if color_choice in ["red", "blue", "green", "orange", "purple"]
                    else defaults["ray_color"]
                )
                params["fps"] = int(
                    input(f"Enter FPS [{defaults['fps']}]: ") or defaults["fps"]
                )
                params["arc_angle"] = defaults["arc_angle"]
            except ValueError:
                print("Invalid input, using defaults.")
                params = defaults.copy()

        params["save_animation"] = (
            input("Save animation as GIF? (y/n): ").strip().lower() == "y"
        )
        filename = None
        if params["save_animation"]:
            input_filename = input("Enter filename (or press Enter for auto): ").strip()
            if input_filename:
                if not input_filename.endswith(".gif"):
                    input_filename += ".gif"
                filename = input_filename

        params["filename"] = filename

        print(f"\nStarting animation with {params['N_round_trips']} round trips...")
        print("Cavity parameters:")
        self.print_info(
            params["y0_initial"], params["theta0_initial_deg"], params["N_round_trips"]
        )

        if not params["save_animation"]:
            print("Close the plot window to continue.")

        fig, anim = self.animate_cavity(**params)
        return fig, anim


def main():
    """Main function for interactive animated ray tracing."""

    use_default_cavity = (
        input("Use default cavity parameters? (R1=-80, R2=-80, L=70) (y/n): ")
        .strip()
        .lower()
        == "y"
    )

    if use_default_cavity:
        R1, R2, L = -80.0, -80.0, 70.0
    else:
        try:
            R1 = float(input("Enter R1 (- for concave, + for convex) [-80]: ") or -80.0)
            R2 = float(input("Enter R2 (- for concave, + for convex) [-80]: ") or -80.0)
            L = float(input("Enter cavity length L [70]: ") or 70.0)
        except ValueError:
            print("Invalid input, using defaults.")
            R1, R2, L = -80.0, -80.0, 70.0

    try:
        animator = AnimateCavityRayTracing(R1=R1, R2=R2, L=L)
        fig, anim = animator.run_interactive()
        return fig, anim
    except Exception as e:
        print(f"Error: {e}")
        return None, None

<a id="73-animation-saving"></a>
### **7.3 Animation Saving**

Animations are exported as GIF files into `OUTPUTS/ANIMATIONS` via `anim.save(..., writer="ffmpeg")`. This requires **FFmpeg** to be installed and on the system `PATH`. If FFmpeg is unavailable, Matplotlib's `PillowWriter` is a drop-in alternative (`writer="pillow"`), provided the Pillow package is installed. Export is wrapped in error handling, so a missing writer produces a readable message rather than a broken cell. Saving a long animation renders every frame and can take noticeably longer than displaying it.

<a id="8-predefined-cavity-configurations"></a>
## 8. Predefined Cavity Configurations

`CAVITY_CASES` collects ready-made parameter sets used by the comparison plots and the batch animation export. Each case bundles the cavity geometry ($R_1$, $R_2$, $L$) with launch conditions and display settings.

<a id="81-concave-concave-cavities"></a>
### **8.1 Concave-Concave Cavities**

Both mirrors focus, so this family contains the classic stable resonators:

- **CC01 (confocal)** — $L = |R_1| = |R_2| = 80$ cm, giving $g_1 = g_2 = 0$ (a marginal boundary case, see 8.3).
- **CC02 (near-concentric)** — a symmetric cavity with $g_1 g_2 = -0.36$, comfortably inside the stable region.
- **CC03 (near-concentric)** — an asymmetric cavity with different radii, $g_1 g_2 \approx 0.17$; the ray pattern loses its mirror symmetry but stays bounded.
- **CC04 (concentric side)** — $g_1 g_2 \approx 0.07$, asymmetric and concentric-side, though not a clean symmetric near-concentric case.

<a id="82-concave-convex-cavities"></a>
### **8.2 Concave-Convex Cavities**

One mirror focuses while the other defocuses, so the stability window is narrower: the concave mirror must overcompensate the divergence added by the convex one. The four cases (CX01–CX04) sample this competition, with stability products ranging from $\approx 0.87$ (close to the $g_1g_2 = 1$ boundary) down to $\approx 0.12$.

<a id="83-unstable-and-marginal-cases"></a>
### **8.3 Unstable and Marginal Cases**

Two boundary situations are worth flagging explicitly:

- **CC01 (confocal)** sits exactly on the $g_1g_2 = 0$ boundary — a *marginal* case. The code's inclusive check reports it as stable, and rays retrace highly symmetric closed paths, but any small length error moves it off the boundary.
- **CX01** with $g_1g_2 \approx 0.87$ is close to the $g_1g_2 = 1$ boundary: rays remain bounded but refocus weakly, and small parameter changes can tip the cavity unstable.

Truly unstable examples (e.g. any convex–convex cavity, where $g_1g_2 > 1$ always) show ray paths that fan outward and escape the mirror apertures; try one in Section 13 with `visualize_single_ray`.

In [ ]:
CAVITY_CASES = {
    "concave_concave": {
        "CC01_symmetric_confocal_boundary": {
            "R1": -80.0,
            "R2": -80.0,
            "L": 80.0,
            "y0_initial": 15.0,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 10,
            "arc_angle": 30.0,
            "ray_color": "red",
        },
        "CC02_symmetric_near_concentric": {
            "R1": -50.0,
            "R2": -50.0,
            "L": 80.0,
            "y0_initial": 12.0,
            "theta0_initial_deg": 1.5,
            "N_round_trips": 40,
            "arc_angle": 40.0,
            "ray_color": "blue",
        },
        "CC03_asymmetric_near_concentric": {
            "R1": -50.0,
            "R2": -80.0,
            "L": 95.0,
            "y0_initial": -12.0,
            "theta0_initial_deg": 1.0,
            "N_round_trips": 32,
            "arc_angle": 30.0,
            "ray_color": "green",
        },
        "CC04_asymmetric_concentric_side": {
            "R1": -85.0,
            "R2": -40.0,
            "L": 90.0,
            "y0_initial": 6.0,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 30,
            "arc_angle": 25.0,
            "ray_color": "purple",
        },
    },
    "concave_convex": {
        "CX01_near_upper_stability_boundary": {
            "R1": -140.0,
            "R2": 70.0,
            "L": 85.0,
            "y0_initial": 5.0,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 30,
            "arc_angle": 20.0,
            "ray_color": "orange",
        },
        "CX02_focused_concave_convex": {
            "R1": -100.0,
            "R2": 50.0,
            "L": 90.0,
            "y0_initial": 5.5,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 35,
            "arc_angle": 25.0,
            "ray_color": "red",
        },
        "CX03_focused_convex_concave": {
            "R1": 50.0,
            "R2": -100.0,
            "L": 95.0,
            "y0_initial": -6.5,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 40,
            "arc_angle": 35.0,
            "ray_color": "blue",
        },
        "CX04_low_product_convex_concave": {
            "R1": 60.0,
            "R2": -120.0,
            "L": 115.0,
            "y0_initial": 6.0,
            "theta0_initial_deg": 0.0,
            "N_round_trips": 30,
            "arc_angle": 30.0,
            "ray_color": "green",
        },
    },
}


<a id="9-static-comparison-plots"></a>
## 9. Static Comparison Plots

`plot_cavity_cases` arranges four cavities per category in a $2\times 2$ subplot grid, so resonators with different stability products can be compared side by side: tight refocusing, slow beam walk, asymmetric bounce patterns, and near-boundary behavior all become visible at a glance.

`_draw_case` renders one case onto an existing subplot axis. Because it draws into a shared grid (rather than creating its own figure), it uses `cavity._get_mirror_arcs` directly instead of the figure-creating helper of Section 6.

Generated figures are saved in:

```text
OUTPUTS/PLOTS
```

when `save_figures=True`. Called with `choice=None`, the function prompts interactively; the demo cells in Section 13 pass `choice="both"` for reproducible execution.

In [ ]:
def _draw_case(ax, case_name, case_parameters):
    """Draw one configured cavity case on an existing Matplotlib axis."""

    cavity = CavityRayTracing(
        R1=case_parameters["R1"],
        R2=case_parameters["R2"],
        L=case_parameters["L"],
    )

    ray_path, _ = cavity.trace_single_ray(
        y0_initial=case_parameters["y0_initial"],
        theta0_initial_deg=case_parameters["theta0_initial_deg"],
        N_round_trips=case_parameters["N_round_trips"],
    )
    left_mirror, right_mirror = cavity._get_mirror_arcs(case_parameters["arc_angle"])

    ray_color = case_parameters["ray_color"]
    ax.set_facecolor(cavity.color_dict[ray_color]["ax_face"])
    ax.add_patch(left_mirror)
    ax.add_patch(right_mirror)

    ax.plot(
        [-cavity.L / 2, cavity.L / 2],
        [0, 0],
        color="grey",
        lw=1,
        ls="--",
        zorder=1,
        alpha=0.6,
    )

    path_x, path_y = zip(*ray_path)
    ax.plot(
        path_x,
        path_y,
        "-o",
        color=ray_color,
        markersize=2.5,
        linewidth=1.5,
        zorder=3,
    )

    params = cavity.get_cavity_parameters()
    status = "STABLE" if params.is_stable else "UNSTABLE"
    if params.is_confocal:
        title_kind = "Confocal"
    elif params.is_symmetric:
        title_kind = f"Symmetric {params.mirror_description}"
    else:
        title_kind = f"Asymmetric {params.mirror_description}"

    ax.set_title(f"{title_kind} {status}", fontsize=12, pad=8)
    ax.text(
        0.5,
        0.98,
        (
            f"$R_1$ = {params.R1:.1f} cm, "
            f"$R_2$ = {params.R2:.1f} cm, "
            f"$L$ = {params.L:.1f} cm\n"
            f"$g_1$ = {params.g1:.3f}, "
            f"$g_2$ = {params.g2:.3f}, "
            f"$g_1 \\times g_2$ = {params.stability_product:.3f}\n"
            f"$y_0$ = {case_parameters['y0_initial']:.1f} cm, "
            f"$\\theta_0$ = {case_parameters['theta0_initial_deg']:.1f}°, "
            f"Trips = {case_parameters['N_round_trips']}"
        ),
        transform=ax.transAxes,
        va="top",
        ha="center",
        fontsize=8,
        family="monospace",
        bbox={
            "boxstyle": "round",
            "facecolor": cavity.color_dict[ray_color]["params_bg"],
            "alpha": 0.45,
        },
        zorder=6,
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    return cavity


def plot_cavity_cases(
    cases=None,
    save_figures=False,
    output_dir=PLOT_DIR,
    show=True,
    choice=None,
):
    """Plot concave-concave and/or concave-convex cases in separate 2x2 figures.

    Parameters
    ----------
    cases : dict or None
        Dictionary with the same structure as ``CAVITY_CASES``. If omitted,
        the four built-in cases for each cavity category are used.
    save_figures : bool
        Save composite figures as PNG files when ``True``.
    output_dir : str or pathlib.Path
        Directory used when ``save_figures`` is enabled.
    show : bool
        Call ``plt.show()`` after creating the figures.
    choice : str or None
        Which cases to plot. Options: 'concave_concave', 'concave_convex', 'both', or None.
        If None, the user will be prompted to choose interactively.

    Returns
    -------
    tuple
        ``(concave_concave_figure, concave_convex_figure)``. If a case category is
        not selected, its respective figure in the returned tuple will be ``None``.
    """

    if choice is None:
        print("Which cavity cases would you like to plot?")
        print("1. Concave-Concave Cavities")
        print("2. Concave-Convex Cavities")
        print("3. Both")
        user_input = input("Enter choice (1, 2, or 3) [3]: ").strip()
        if user_input == "1":
            choice = "concave_concave"
        elif user_input == "2":
            choice = "concave_convex"
        else:
            choice = "both"

    selected_cases = CAVITY_CASES if cases is None else cases
    figures = {}

    all_specs = {
        "concave_concave": (
            "Concave-Concave Cavities",
            "concave_concave_2x2.png",
        ),
        "concave_convex": (
            "Concave-Convex Cavities",
            "concave_convex_2x2.png",
        ),
    }

    if choice == "concave_concave":
        figure_specs = {"concave_concave": all_specs["concave_concave"]}
    elif choice == "concave_convex":
        figure_specs = {"concave_convex": all_specs["concave_convex"]}
    else:
        figure_specs = all_specs

    for category, (figure_title, filename) in figure_specs.items():
        category_cases = selected_cases[category]
        if len(category_cases) != 4:
            raise ValueError(
                f"{category} must contain exactly four cases for a 2x2 figure"
            )

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.subplots_adjust(
            left=0.03,
            right=0.97,
            top=0.92,
            bottom=0.03,
            hspace=0.15,
            wspace=0.08,
        )
        fig.patch.set_facecolor("black")
        fig.suptitle(figure_title, fontsize=18, fontweight="bold")

        for ax, (case_name, case_parameters) in zip(axes.flat, category_cases.items()):
            _draw_case(ax, case_name, case_parameters)

        figures[category] = fig

        if save_figures:
            save_directory = Path(output_dir)

            choice_prompt = f"For {figure_title}, save as default '{filename}' (d) or enter custom filename (c)? [d]: "
            filename_choice = input(choice_prompt).strip().lower()

            final_filename = filename
            if filename_choice == "c":
                custom_name = input(
                    "Enter custom filename (e.g. custom_name.png): "
                ).strip()
                if custom_name:
                    if not custom_name.endswith(".png"):
                        custom_name += ".png"
                    final_filename = custom_name
            else:
                base_path = save_directory / final_filename
                if base_path.exists():
                    stem = base_path.stem
                    ext = base_path.suffix
                    counter = 1
                    while True:
                        new_filename = f"{stem}_{counter}{ext}"
                        new_path = save_directory / new_filename
                        if not new_path.exists():
                            final_filename = new_filename
                            break
                        counter += 1

            save_path = save_directory / final_filename
            export_figure(fig, save_path)

    if show and figures:
        plt.show()

    return figures.get("concave_concave"), figures.get("concave_convex")

<a id="10-stability-diagram"></a>
## 10. Stability Diagram

<a id="101-stability-region-in-the-g-plane"></a>
### **10.1 Stability Region in the g Plane**

The two-mirror stability condition can be visualized in the plane of $g_1$ and $g_2$. The implementation in this notebook shades the region satisfying the inclusive condition

$$
0 \le g_1 g_2 \le 1
$$

matching the `is_stable` check in `CavityRayTracing`. Strictly, the boundaries $g_1g_2 = 0$ (the coordinate axes) and $g_1g_2 = 1$ (the hyperbola branches and the point $g_1 = g_2 = 1$) are **marginal** rather than robustly stable — the notebook's convention is to shade them as part of the stable region but label them explicitly as boundaries.

<a id="102-interpreting-the-diagram"></a>
### **10.2 Interpreting the Diagram**

- Points **inside** the shaded region correspond to bounded paraxial ray motion.
- Points **on the boundary curves** ($g_1g_2 = 0$ or $g_1g_2 = 1$) are marginal cases — the confocal ($0,0$), planar ($1,1$), concentric ($-1,-1$), and hemispherical ($1,0$ / $0,1$) cavities all sit exactly on these boundaries.
- Points **outside** the region correspond to unstable resonators.
- A selected cavity is plotted as a marker at its computed $(g_1, g_2)$ coordinate — green if stable, red if unstable.

<a id="103-stability-diagram-implementation"></a>
### **10.3 Stability Diagram Implementation**

`plot_stability_diagram` shades the stable region, draws the marginal boundary curves, labels the stable and unstable regions, annotates the classic configurations, and marks the operating point of an optional `CavityRayTracing` instance. Pass `save_figure=True` to export the figure to `OUTPUTS/PLOTS`.

In [ ]:
def plot_stability_diagram(cavity=None, save_figure=False, filename=None):
    """
    Plot the standard g1-g2 stability diagram for two-mirror optical cavities.

    Shades the stable region (0 <= g1*g2 <= 1), draws the marginal boundary
    curves (g1*g2 = 0 and g1*g2 = 1), labels stable/unstable regions, and
    annotates the classic cavity configurations.

    Parameters
    ----------
    cavity : CavityRayTracing or None
        If provided, its operating point (g1, g2) is plotted and color-coded
        (green = stable, red = unstable).
    save_figure : bool
        Save the figure as PNG into OUTPUTS/PLOTS when True.
    filename : str or None
        Filename for the saved figure (auto-generated if None).

    Returns
    -------
    fig, ax : matplotlib figure and axes
    """
    g_vals = np.linspace(-3, 3, 400)
    G1, G2 = np.meshgrid(g_vals, g_vals)

    # Stability condition: 0 <= g1*g2 <= 1
    stability_product = G1 * G2
    stable_region = (stability_product >= 0) & (stability_product <= 1)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor("#00001A")
    fig.patch.set_facecolor("black")

    ax.contourf(G1, G2, stable_region, levels=[0.5, 1.5], colors=["#00ffcc"], alpha=0.2)

    # Marginal boundary curve: g1 * g2 = 1 (hyperbola)
    g_curve_pos = np.linspace(0.01, 3, 200)
    g_curve_neg = np.linspace(-3, -0.01, 200)
    ax.plot(g_curve_pos, 1 / g_curve_pos, color="white", lw=1.5, ls="--")
    ax.plot(g_curve_neg, 1 / g_curve_neg, color="white", lw=1.5, ls="--")

    # Marginal boundary curves: g1 * g2 = 0 (the axes)
    ax.axhline(0, color="white", lw=1.5)
    ax.axvline(0, color="white", lw=1.5)

    region_style = dict(fontsize=11, fontweight="bold", ha="center", va="center")
    ax.text(0.55, 0.75, "STABLE", color="#00ffcc", **region_style)  # type: ignore
    ax.text(-0.55, -0.75, "STABLE", color="#00ffcc", **region_style)  # type: ignore
    ax.text(-1.8, 2.2, "UNSTABLE", color="#ff6666", **region_style)  # type: ignore
    ax.text(1.8, -2.2, "UNSTABLE", color="#ff6666", **region_style)  # type: ignore
    ax.text(
        2.35,
        2.35,
        "UNSTABLE\n($g_1g_2 > 1$)",
        color="#ff6666",
        fontsize=10,
        fontweight="bold",
        ha="center",
        va="center",
    )
    ax.text(
        2.2,
        0.62,
        "$g_1g_2 = 1$ (marginal)",
        color="#cccccc",
        fontsize=10,
        rotation=-18,
        ha="center",
    )
    ax.text(
        -2.35,
        0.12,
        "$g_1g_2 = 0$ (marginal)",
        color="#cccccc",
        fontsize=10,
        ha="center",
        va="bottom",
    )

    marker_style = dict(color="white", marker="+", markersize=10)
    ax.plot(1, 1, **marker_style)  # type: ignore
    ax.text(1.1, 1.1, "Planar", color="#cccccc", fontsize=10)

    ax.plot(0, 0, **marker_style)  # type: ignore
    ax.text(0.1, 0.1, "Confocal", color="#cccccc", fontsize=10)

    ax.plot(-1, -1, **marker_style)  # type: ignore
    ax.text(-0.9, -1.2, "Concentric", color="#cccccc", fontsize=10)

    ax.plot(1, 0, **marker_style)  # type: ignore
    ax.plot(0, 1, **marker_style)  # type: ignore
    ax.text(1.1, 0.1, "Hemispherical", color="#cccccc", fontsize=10)

    if cavity is not None:
        params = cavity.get_cavity_parameters()
        g1, g2 = params.g1, params.g2

        point_color = "#00ff00" if params.is_stable else "#ff3333"
        label_text = (
            f"Current Cavity\n"
            f"($g_1$={g1:.2f}, $g_2$={g2:.2f}, "
            f"$g_1g_2$={params.stability_product:.2f})"
        )

        ax.plot(
            g1,
            g2,
            marker="o",
            markersize=10,
            color=point_color,
            markeredgecolor="white",
            markeredgewidth=2,
            label=label_text,
        )

        ax.legend(
            facecolor="black", edgecolor="white", labelcolor="white", loc="upper left"
        )

    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")

    ax.set_title("Stability Diagram ($g_1-g_2$ Plane)", color="white", pad=20)
    ax.set_xlabel("$g_1 = 1 + L/R_1$", color="white", fontsize=14)
    ax.set_ylabel("$g_2 = 1 + L/R_2$", color="white", fontsize=14)

    for spine in ax.spines.values():
        spine.set_color("white")
    ax.tick_params(colors="white")
    ax.grid(color="white", alpha=0.1)

    if save_figure:
        if filename is None:
            filename = f"stability_diagram_{timestamp}.png"
        elif not filename.endswith(".png"):
            filename += ".png"
        export_figure(fig, PLOT_DIR / filename)

    plt.show()
    return fig, ax

<a id="11-symmetric-confocal-optical-cavity"></a>
## 11. Symmetric Confocal Optical Cavity

This section focuses on the special case

$$
L = |R_1| = |R_2|
$$

<a id="111-confocal-stability"></a>
### **11.1 Confocal Stability**

With the notebook sign convention (concave mirrors), the symmetric confocal cavity has

$$
R_1 = R_2 = -L
$$

so that

$$
g_1 = g_2 = 0
$$

This places the confocal cavity exactly at the origin of the stability diagram — a **marginal boundary case**, not an interior point of the stable region. Ideal confocal geometry is therefore a limiting case: any small deviation of $L$ from $|R|$ moves the operating point off the boundary. It is nevertheless extremely useful pedagogically, because the two mirror focal points coincide at the cavity center and paraxial rays **retrace closed, highly symmetric paths** — a ray launched parallel to the axis returns to its starting point after two round trips.

<a id="112-static-confocal-ray-plot"></a>
### **11.2 Static Confocal Ray Plot**

`plot_confocal_ray_tracing` draws each mirror-to-mirror segment in a different color (cycling through four), adds direction arrows to the first four segments, and marks the centers of curvature ($C_1$, $C_2$) and the focal point(s) on the optical axis. For a true confocal cavity the two foci coincide at a single point $F$ at the cavity center. The canvas is built with the Section 6 helpers.

In [ ]:
def plot_confocal_ray_tracing(
    cavity,
    y0_initial=15.0,
    theta0_initial_deg=0.0,
    N_round_trips=2,
    arc_angle=25.0,
    save_figure=False,
    filename=None,
):
    """
    Visualize the cavity with ray tracing using multiple colors for segments.

    Parameters
    ----------
    cavity : CavityRayTracing
        The cavity object containing all physical parameters and methods (e.g., trace_ray).
    y0_initial : float
        Initial height of ray from optical axis
    theta0_initial_deg : float
        Initial angle in degrees
    N_round_trips : int
        Number of round trips to simulate
    arc_angle : float
        Angle for arc representation of mirrors (0-90 degrees)
    save_figure : bool
        Whether to save the figure as PNG into OUTPUTS/PLOTS
    filename : str or None
        Filename for saved figure (auto-generated if None)

    Returns
    -------
    fig, ax : matplotlib figure and axes
    """
    if not 0 < arc_angle < 90:
        raise ValueError("arc_angle must be between 0 and 90 degrees")

    try:
        ray_segments, _ = cavity.trace_ray(
            y0_initial, theta0_initial_deg, N_round_trips
        )
    except Exception as e:
        raise ValueError(f"Ray tracing failed: {str(e)}")

    fig, ax = setup_cavity_axes(
        cavity,
        arc_angle=arc_angle,
        figsize=(12, 8),
        mirror_linewidth=8,
        axis_alpha=0.5,
    )
    fig.subplots_adjust(left=0.01, right=0.84, top=0.94, bottom=0.02)

    colors = ["red", "blue", "green", "orange"]

    # Centers of curvature
    ax.scatter(
        [cavity.C1x, cavity.C2x], [0, 0], s=60, marker="x", color="white", zorder=7
    )
    ax.annotate(
        "C1",
        (cavity.C1x, 0),
        textcoords="offset points",
        xytext=(8, 0),
        ha="left",
        va="center",
        fontsize=10,
        color="white",
        zorder=7,
    )
    ax.annotate(
        "C2",
        (cavity.C2x, 0),
        textcoords="offset points",
        xytext=(-8, 0),
        ha="right",
        va="center",
        fontsize=10,
        color="white",
        zorder=7,
    )

    # Foci
    tol = 1e-6
    if abs(cavity.F1x - cavity.F2x) < tol:
        # Single focus point (confocal case)
        ax.scatter([cavity.F1x], [0], s=50, marker="o", color="magenta", zorder=8)
        ax.annotate(
            "F",
            (cavity.F1x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=10,
            color="magenta",
        )
    else:
        # Two separate foci
        ax.scatter(
            [cavity.F1x, cavity.F2x],
            [0, 0],
            s=50,
            marker="o",
            color="magenta",
            zorder=8,
        )
        ax.annotate(
            "F1",
            (cavity.F1x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=10,
            color="magenta",
        )
        ax.annotate(
            "F2",
            (cavity.F2x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=10,
            color="magenta",
        )

    for i, segment in enumerate(ray_segments):
        if len(segment) < 2:
            continue

        path_x, path_y = zip(*segment)
        color = colors[i % 4]

        if i % 2 == 0:
            direction = "M1→M2"
        else:
            direction = "M2→M1"

        if i < 4:
            segment_label = f"Segment {i + 1} ({direction})"
        else:
            segment_label = None

        ax.plot(
            path_x,
            path_y,
            color=color,
            marker="o",
            markersize=4,
            linestyle="-",
            linewidth=2,
            label=segment_label,
            zorder=3,
            alpha=0.8,
        )

        if len(path_x) >= 2 and i < 4:
            x0, y0 = path_x[-2], path_y[-2]
            x1, y1 = path_x[-1], path_y[-1]

            # Place arrow at 85% along segment
            frac = 0.85
            xa, ya = x0 + frac * (x1 - x0), y0 + frac * (y1 - y0)

            arrow = FancyArrowPatch(
                (xa, ya),
                (x1, y1),
                connectionstyle="arc3",
                arrowstyle="-|>",
                mutation_scale=28,
                color=color,
                linewidth=2,
                zorder=6,
            )
            ax.add_patch(arrow)

    # Cavity status
    cavity_type = (
        "CONFOCAL"
        if cavity.is_confocal
        else ("STABLE" if cavity.is_stable else "UNSTABLE")
    )

    param_text = (
        f"Cavity Parameters\n"
        f"{'-' * 20}\n"
        f"$R_1$ = {cavity.R1:.1f} cm\n"
        f"$R_2$ = {cavity.R2:.1f} cm\n"
        f"$L$ = {cavity.L:.1f} cm\n"
        f"$g_1$ = {cavity.g1:.3f}\n"
        f"$g_2$ = {cavity.g2:.3f}\n"
        f"$g_1 \\times g_2$ = {cavity.stability_product:.3f}\n"
        f"Initial θ = {theta0_initial_deg:.1f}°\n"
        f"Round trips: {N_round_trips}\n"
        f"Total segments: {len(ray_segments)}"
    )
    add_cavity_parameter_box(ax, text=param_text)

    ax.set_title(f"{cavity_type} Cavity Ray Tracing", pad=10)
    ax.legend(loc="upper center", fontsize=9)

    if save_figure:
        if filename and not filename.endswith(".png"):
            filename += ".png"
        elif not filename:
            metadata = (
                f"R1_{cavity.R1}_R2_{cavity.R2}_L_{cavity.L}_theta_{theta0_initial_deg}"
            )
            filename = f"confocal_cavity_{metadata}_{timestamp}.png"
        export_figure(fig, PLOT_DIR / filename)

    plt.show()
    return fig, ax

<a id="113-confocal-ray-animation"></a>
### **11.3 Confocal Ray Animation**

`animate_confocal_ray_tracing` animates the same segment-colored picture: the shared `interpolate_segments` helper (Section 4) subdivides each chord, completed segments are frozen in their color, and a moving arrowhead tracks the ray front while a counter reports round trips. Set `save_animation=True` to export a GIF to `OUTPUTS/ANIMATIONS` (see Section 7.3 for writer requirements).

In [ ]:
def animate_confocal_ray_tracing(
    cavity,
    y0_initial=15.0,
    theta0_initial_deg=0.0,
    N_round_trips=2,
    arc_angle=25.0,
    fps=30,
    points_per_segment=50,
    save_animation=False,
    filename=None,
):
    """
    Animate the confocal cavity ray tracing with colored segments.

    Parameters
    ----------
    cavity : CavityRayTracing
        The cavity object containing all physical parameters and methods.
    y0_initial : float
        Initial height of ray from optical axis
    theta0_initial_deg : float
        Initial angle in degrees
    N_round_trips : int
        Number of round trips to simulate
    arc_angle : float
        Angle for arc representation of mirrors
    fps : int
        Frames per second for animation
    points_per_segment : int
        Number of interpolated points between mirror reflections
    save_animation : bool
        Whether to save the animation as GIF
    filename : str or None
        Filename for saved animation (auto-generated if None)

    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object
    anim : matplotlib.animation.FuncAnimation
        The animation object
    """
    if not 0 < arc_angle < 90:
        raise ValueError("arc_angle must be between 0 and 90 degrees")

    try:
        ray_segments, _ = cavity.trace_ray(
            y0_initial, theta0_initial_deg, N_round_trips
        )
    except Exception as e:
        raise ValueError(f"Ray tracing failed: {str(e)}")

    all_points, segment_indices, point_indices_in_segment, npps = interpolate_segments(
        ray_segments, points_per_segment
    )
    segment_colors = ["red", "blue", "green", "orange"]

    fig, ax = setup_cavity_axes(
        cavity,
        arc_angle=arc_angle,
        figsize=(12, 8),
        mirror_linewidth=6,
        axis_alpha=0.8,
    )
    fig.subplots_adjust(top=0.92, bottom=0.04, left=0.02, right=0.82)

    timer_bg = (
        getattr(cavity, "color_dict", {}).get("blue", {}).get("timer_bg", "white")
    )

    # Centers of curvature
    ax.scatter(
        [cavity.C1x, cavity.C2x], [0, 0], s=60, marker="x", color="white", zorder=7
    )
    ax.annotate(
        "C1",
        (cavity.C1x, 0),
        textcoords="offset points",
        xytext=(8, 0),
        ha="left",
        va="center",
        fontsize=12,
        color="white",
        zorder=7,
    )
    ax.annotate(
        "C2",
        (cavity.C2x, 0),
        textcoords="offset points",
        xytext=(-8, 0),
        ha="right",
        va="center",
        fontsize=12,
        color="white",
        zorder=7,
    )

    # Foci
    tol = 1e-6
    if abs(cavity.F1x - cavity.F2x) < tol:
        ax.scatter([cavity.F1x], [0], s=50, marker="o", color="magenta", zorder=8)
        ax.annotate(
            "F",
            (cavity.F1x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=12,
            color="magenta",
        )
    else:
        ax.scatter(
            [cavity.F1x, cavity.F2x],
            [0, 0],
            s=50,
            marker="o",
            color="magenta",
            zorder=12,
        )
        ax.annotate(
            "F1",
            (cavity.F1x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=12,
            color="magenta",
        )
        ax.annotate(
            "F2",
            (cavity.F2x, 0),
            textcoords="offset points",
            xytext=(0, -12),
            ha="center",
            va="top",
            fontsize=12,
            color="magenta",
        )

    if getattr(cavity, "is_confocal", False) and cavity.is_stable:
        cavity_type = "CONFOCAL STABLE"
    elif cavity.is_stable:
        cavity_type = "STABLE"
    else:
        cavity_type = "UNSTABLE"

    param_text = (
        f"{cavity_type} Cavity\n"
        f"{'-' * (len(cavity_type) + 7)}\n"
        f"$R_1$ = {cavity.R1:.1f} cm\n"
        f"$R_2$ = {cavity.R2:.1f} cm\n"
        f"$L$ = {cavity.L:.1f} cm\n"
        f"$g_1$ = {cavity.g1:.3f}\n"
        f"$g_2$ = {cavity.g2:.3f}\n"
        f"Initial $\\theta$ = {theta0_initial_deg:.1f}°"
    )
    add_cavity_parameter_box(ax, text=param_text)

    progress_text = ax.text(
        0.5,
        0.98,
        f"Round trips: 0/{N_round_trips}",
        transform=ax.transAxes,
        ha="center",
        va="top",
        bbox=dict(
            boxstyle="round",
            facecolor=timer_bg,
            alpha=0.8,
        ),
        fontsize=12,
        fontweight="bold",
        fontfamily="monospace",
    )

    segment_lines = []
    (current_line,) = ax.plot([], [], "-", linewidth=2, zorder=3, alpha=0.8)

    state = {"arrow_patch": None}
    ax.set_title(f"{cavity_type} Cavity Ray Tracing", fontsize=14, pad=10)

    completed_segments = set()

    def animate(frame):
        if frame >= len(all_points):
            return [current_line, progress_text] + segment_lines

        current_point = all_points[frame]
        x_current, y_current = current_point
        current_segment = segment_indices[frame]
        color = segment_colors[current_segment % 4]

        completed = current_segment // 2
        if (current_segment % 2 == 1) and (point_indices_in_segment[frame] == npps - 1):
            completed += 1

        if completed > N_round_trips:
            completed = N_round_trips

        progress_text.set_text(f"Round trips: {completed}/{N_round_trips}")

        segment_start_idx = None
        segment_end_idx = None
        for i, seg_idx in enumerate(segment_indices):
            if seg_idx == current_segment:
                if segment_start_idx is None:
                    segment_start_idx = i
                segment_end_idx = i

        if frame <= segment_end_idx:
            current_segment_points = all_points[segment_start_idx : frame + 1]
            current_x = [p[0] for p in current_segment_points]
            current_y = [p[1] for p in current_segment_points]

            current_line.set_data(current_x, current_y)
            current_line.set_color(color)

        for seg_idx in range(current_segment):
            if seg_idx not in completed_segments:
                seg_points = [
                    all_points[i] for i, s in enumerate(segment_indices) if s == seg_idx
                ]
                if seg_points:
                    seg_x = [p[0] for p in seg_points]
                    seg_y = [p[1] for p in seg_points]
                    seg_color = segment_colors[seg_idx % 4]

                    (line,) = ax.plot(
                        seg_x,
                        seg_y,
                        "-",
                        linewidth=2,
                        color=seg_color,
                        zorder=3,
                        alpha=0.8,
                    )
                    segment_lines.append(line)
                    completed_segments.add(seg_idx)

        if state["arrow_patch"] is not None:
            state["arrow_patch"].remove()
            state["arrow_patch"] = None

        if frame > 0:
            prev_point = all_points[frame - 1]
            x_prev, y_prev = prev_point

            dx = x_current - x_prev
            dy = y_current - y_prev

            if dx**2 + dy**2 > 1e-6:
                arrow_start_x = x_current - 0.2 * dx
                arrow_start_y = y_current - 0.2 * dy

                state["arrow_patch"] = FancyArrowPatch(  # type: ignore
                    (arrow_start_x, arrow_start_y),
                    (x_current, y_current),
                    connectionstyle="arc3",
                    arrowstyle="-|>",
                    mutation_scale=25,
                    color=color,
                    linewidth=2,
                    zorder=6,
                )
                ax.add_patch(state["arrow_patch"])

        return (
            [current_line, progress_text]
            + segment_lines
            + ([state["arrow_patch"]] if state["arrow_patch"] else [])
        )

    anim = FuncAnimation(
        fig,
        animate,
        frames=len(all_points),
        interval=int(1000 / fps),
        blit=True,
        repeat=False,
    )

    if save_animation:
        if filename and not filename.endswith(".gif"):
            filename += ".gif"
        elif not filename:
            metadata = f"R1_{cavity.R1}_R2_{cavity.R2}_L_{cavity.L}"
            filename = f"confocal_cavity_animation_{metadata}_{timestamp}.gif"

        try:
            filepath = ANIMATION_DIR / filename
            print(f"Saving animation to {filepath}...")
            anim.save(filepath, writer="ffmpeg", fps=fps, dpi=120)
            print("Animation saved successfully!")
        except Exception as e:
            print(f"Error saving animation: {e}")
        plt.close(fig)
    else:
        plt.show()

    return fig, anim

<a id="12-multiple-ray-bundle-tracing"></a>
## 12. Multiple-Ray Bundle Tracing

Why trace many rays instead of one?

- A **single ray** shows one trajectory — informative, but easy to over-interpret.
- A **ray bundle** launched with a spread of angles reveals focusing, spreading, refocusing, and unstable divergence far more clearly: in a stable cavity the bundle stays confined and periodically refocuses, while in an unstable one it fans out and escapes.
- Different launch angles **test the robustness** of the cavity stability, not just one lucky trajectory.

`multiple_ray_trace_cavity` traces `num_rays` rays with launch angles evenly spread up to `max_angle_deg`, colors them with a colormap, and can produce a static plot, an animation, or both. It uses the shared `interpolate_segments` helper for animation frames and the Section 6 canvas helpers for figure setup, and it initializes its return values so an animation can be *saved* without being displayed.

For reproducible notebook execution, pass

```python
interactive_prompt=False
```

as the example cells in Section 13 do — otherwise the function asks display/save questions on the command line, which is useful interactively but blocks unattended runs.

In [ ]:
def multiple_ray_trace_cavity(
    cavity=None,
    N_round_trips=10,
    y0_initial=15.0,
    num_rays=5,
    max_angle_deg=2.0,
    arc_angle=25.0,
    colorscale="plasma",
    use_colormap=True,
    default_color="blue",
    save_fig=False,
    save_ani=False,
    display_plot=True,
    display_ani=False,
    filename=None,
    interactive_prompt=True,
    fps=30,
    points_per_segment=50,
):
    """
    Ray tracing simulation for a bundle of rays in a spherical mirror optical cavity.
    Capable of generating both static plots and animations.

    Parameters
    ----------
    cavity : CavityRayTracing
        The cavity instance to trace rays through.
    N_round_trips, y0_initial, num_rays, max_angle_deg :
        Ray tracing simulation parameters.
    arc_angle : float
        Angle for arc representation of mirrors.
    colorscale, use_colormap, default_color :
        Styling settings.
    save_fig, save_ani : bool
        Whether to save outputs.
    display_plot, display_ani : bool
        Whether to display outputs.
    filename : str or None
        Base filename without extension. Auto-generated if None.
    interactive_prompt : bool
        If True, asks the user via CLI what they want to do.
        Use interactive_prompt=False for reproducible notebook runs.
    fps, points_per_segment :
        Animation parameters.
    """

    fig = None
    anim = None

    # Interactive Prompts
    if interactive_prompt:
        print("\n--- Ray Tracing Display/Save Options ---")
        ans_plot = input("Display static plot? (y/n, default=y): ").strip().lower()
        display_plot = False if ans_plot == "n" else True

        if display_plot:
            ans_save = input("Save static plot? (y/n, default=n): ").strip().lower()
            save_fig = True if ans_save == "y" else save_fig

        ans_ani = (
            input("Generate and display animation? (y/n, default=n): ").strip().lower()
        )
        display_ani = True if ans_ani == "y" else display_ani

        if display_ani:
            ans_s_ani = (
                input("Save animation as GIF? (y/n, default=n): ").strip().lower()
            )
            save_ani = True if ans_s_ani == "y" else save_ani

        if save_fig or save_ani:
            custom_name = input(
                "Enter custom filename base (leave blank for default): "
            ).strip()
            if custom_name:
                filename = custom_name

    # Simulation setup
    if cavity is None:
        raise ValueError("Please provide a CavityRayTracing instance.")

    # Colors for rays
    if use_colormap:
        ray_colors = plt.get_cmap(colorscale)(np.linspace(0, 1, num_rays))
    else:
        ray_colors = [default_color] * num_rays

    initial_angles_deg = np.linspace(0, max_angle_deg, num_rays)

    # Trace all rays
    all_ray_paths_flat = []
    all_interpolated_rays = []

    for theta in initial_angles_deg:
        try:
            ray_segments, _ = cavity.trace_ray(y0_initial, theta, N_round_trips)
        except Exception as e:
            print(f"Warning: Ray tracing failed for angle {theta:.2f}°: {e}")
            continue

        pts = []
        for seg in ray_segments:
            if not pts:
                pts.append(seg[0])
            if len(seg) > 1:
                pts.append(seg[1])
        all_ray_paths_flat.append(pts)

        if display_ani or save_ani:
            interp_pts, _, _, _ = interpolate_segments(ray_segments, points_per_segment)
            all_interpolated_rays.append(interp_pts)

    def setup_canvas():
        fig_canvas, ax_canvas = setup_cavity_axes(
            cavity,
            arc_angle=arc_angle,
            figsize=(12, 8),
            mirror_linewidth=6,
            axis_alpha=0.8,
        )
        fig_canvas.subplots_adjust(top=0.94, bottom=0.02, left=0.02, right=0.82)

        if cavity.is_confocal:
            title_str = "Confocal"
        elif cavity.is_symmetric:
            title_str = f"Symmetric {cavity.mirror_description}"
        else:
            title_str = f"Asymmetric {cavity.mirror_description}"

        param_text = (
            f"Design Parameters\n"
            f"{'-' * 20}\n"
            f"$R_1$ = {cavity.R1:.1f} cm\n"
            f"$R_2$ = {cavity.R2:.1f} cm\n"
            f"$L$ = {cavity.L:.1f} cm\n"
            f"$g_1$ = {cavity.g1:.3f}\n"
            f"$g_2$ = {cavity.g2:.3f}\n"
            f"$g_1 \\times g_2 = {cavity.g1 * cavity.g2:.3f}$\n"
            f"Stability: {'STABLE' if cavity.is_stable else 'UNSTABLE'}\n"
            f"Rays: {num_rays}\n"
            f"Round trips: {N_round_trips}"
        )
        add_cavity_parameter_box(
            ax_canvas,
            text=param_text,
            facecolor="#001A00",
            alpha=1.0,
            y=0.7,
        )

        ax_canvas.set_title(f"Multiple Ray Tracing in {title_str} Cavity", pad=10)

        return fig_canvas, ax_canvas

    # STATIC PLOTTING
    # Base filename generation
    if filename is None:
        filename = (
            f"cavity_bundle_R1_{cavity.R1}_R2_{cavity.R2}_L_{cavity.L}_"
            f"rays_{num_rays}_{timestamp}"
        )

    if display_plot or save_fig:
        fig_s, ax_s = setup_canvas()
        fig = fig_s

        for j, pts in enumerate(all_ray_paths_flat):
            path_x, path_y = zip(*pts)
            label = f"$\\theta_0$ = {initial_angles_deg[j]:.2f}°"

            ax_s.plot(
                path_x,
                path_y,
                color=ray_colors[j],
                marker="o",
                markersize=2,
                linestyle="-",
                label=label,
                alpha=0.7,
                zorder=3,
            )

        ax_s.legend(bbox_to_anchor=(1.01, 0.8))

        if save_fig:
            static_path = PLOT_DIR / f"{filename}.png"
            print("Saving static figure...")
            export_figure(fig_s, static_path)

        if display_plot:
            plt.show(block=(not display_ani))
        else:
            plt.close(fig_s)

    # ANIMATION

    if display_ani or save_ani:
        fig_a, ax_a = setup_canvas()
        fig = fig_a

        max_frames = (
            max(len(pts) for pts in all_interpolated_rays)
            if all_interpolated_rays
            else 0
        )

        ray_lines = []
        for j in range(num_rays):
            label = f"$\\theta_0$ = {initial_angles_deg[j]:.2f}°"
            (line,) = ax_a.plot(
                [], [], "-", color=ray_colors[j], linewidth=2, zorder=3, label=label
            )
            ray_lines.append(line)

        ax_a.legend(bbox_to_anchor=(1.01, 0.8))
        timer_bg = (
            getattr(cavity, "color_dict", {}).get("blue", {}).get("timer_bg", "white")
        )
        progress_text = ax_a.text(
            0.5,
            0.98,
            "Progress: 0%",
            transform=ax_a.transAxes,
            ha="center",
            va="top",
            bbox=dict(boxstyle="round", facecolor=timer_bg, alpha=0.8),
            fontsize=12,
            fontweight="bold",
            fontfamily="monospace",
        )

        arrow_patches = np.array([None] * num_rays, dtype=object)

        def animate(frame):
            for i, p in enumerate(arrow_patches):
                if p is not None:
                    p.remove()
                    arrow_patches[i] = None

            for j, pts in enumerate(all_interpolated_rays):
                if frame < len(pts):
                    current_segment_points = pts[: frame + 1]
                    current_x = [p[0] for p in current_segment_points]
                    current_y = [p[1] for p in current_segment_points]
                    ray_lines[j].set_data(current_x, current_y)

                    if frame > 0:
                        prev_point = pts[frame - 1]
                        curr_point = pts[frame]
                        dx = curr_point[0] - prev_point[0]
                        dy = curr_point[1] - prev_point[1]

                        if dx**2 + dy**2 > 1e-6:
                            arrow_start_x = curr_point[0] - 0.2 * dx
                            arrow_start_y = curr_point[1] - 0.2 * dy

                            arrow = FancyArrowPatch(
                                (arrow_start_x, arrow_start_y),
                                (curr_point[0], curr_point[1]),
                                connectionstyle="arc3",
                                arrowstyle="-|>",
                                mutation_scale=15,
                                color=ray_colors[j],
                                zorder=6,
                            )
                            ax_a.add_patch(arrow)
                            arrow_patches[j] = arrow

            pct = int((frame / max_frames) * 100) if max_frames > 0 else 100
            progress_text.set_text(f"Progress: {pct}%")

            valid_patches = [p for p in arrow_patches if p is not None]
            return ray_lines + [progress_text] + valid_patches

        anim = FuncAnimation(
            fig_a,
            animate,
            frames=max_frames,
            interval=int(1000 / fps),
            blit=True,
            repeat=False,
        )

        if save_ani:
            ani_path = ANIMATION_DIR / f"{filename}.gif"
            try:
                print("Saving animation...")
                anim.save(ani_path, writer="ffmpeg", fps=fps, dpi=120)
                print("Animation saved successfully!")
            except Exception as e:
                print(f"Error saving animation: {e}")
            plt.close(fig_a)
        else:
            plt.show()

    return fig, anim

<a id="13-example-studies-and-outputs"></a>
## 13. Example Studies and Outputs

All demonstration cells are collected here, separated from the definition cells above. Each example sets `interactive_prompt=False` (or an explicit `choice`) where applicable, so the whole notebook can be executed top-to-bottom without manual input. Static plots go to `OUTPUTS/PLOTS` and animations to `OUTPUTS/ANIMATIONS` whenever saving is enabled.

### **13.1 Stable Symmetric Concave-Concave Cavity — Single Ray**

A symmetric concave–concave cavity with $R_1 = R_2 = -80$ cm and $L = 70$ cm has $g_1 = g_2 = 0.125$ and $g_1g_2 \approx 0.016$ — deep inside the stable region. The single ray remains bounded over 50 round trips, weaving a confined pattern between the mirrors.

In [ ]:
# Stable symmetric concave-concave cavity, single ray
cavity_cc = CavityRayTracing(R1=-80.0, R2=-80.0, L=70.0)
cavity_cc.print_info(y0_initial=15.5, theta0_initial_deg=0.0, N_round_trips=50)
fig_cc, ax_cc = cavity_cc.visualize_single_ray(
    y0_initial=15.5,
    theta0_initial_deg=0.0,
    N_round_trips=50,
    ray_color="red",
    save_figure=False,
)

### **13.2 Static Comparison Grids**

The four concave–concave cases (confocal, near-planar regime, asymmetric, near-concentric) and the four concave–convex cases side by side. Passing `choice="both"` skips the interactive menu; set `save_figures=True` to export the grids to `OUTPUTS/PLOTS`.

In [ ]:
# 2x2 comparison grids for both cavity families
fig_grid_cc, fig_grid_cx = plot_cavity_cases(
    choice="both", save_figures=False, show=True
)

### **13.3 Confocal Cavity — Static Segment Plot**

The symmetric confocal cavity ($R_1 = R_2 = -80$ cm, $L = 80$ cm, so $g_1 = g_2 = 0$). Each mirror-to-mirror segment gets its own color; the coincident focal point $F$ and the centers of curvature $C_1$, $C_2$ are marked on the axis. Note how the ray closes on itself after two round trips.

In [ ]:
confocal_cavity = CavityRayTracing(R1=-80, R2=-80, L=80)
fig_cf, ax_cf = plot_confocal_ray_tracing(confocal_cavity, N_round_trips=2)

### **13.4 Confocal Cavity — Animation**

The same confocal geometry, animated: the ray front advances segment by segment with a moving arrowhead. Set `save_animation=True` to export a GIF to `OUTPUTS/ANIMATIONS` instead of displaying (requires FFmpeg, see Section 7.3).

In [ ]:
fig_cfa, anim_cf = animate_confocal_ray_tracing(
    confocal_cavity,
    y0_initial=15,
    theta0_initial_deg=0,
    N_round_trips=2,
    save_animation=False,
)

### **13.5 Concave-Convex Cavity — Single-Ray Animation**

A stable concave–convex cavity (CX02: $R_1 = -100$ cm, $R_2 = +50$ cm, $L = 90$ cm, $g_1g_2 = 0.28$). The concave mirror's refocusing dominates the convex mirror's defocusing, so the ray stays bounded despite the asymmetric geometry.

In [ ]:
animator_cx = AnimateCavityRayTracing(R1=-100.0, R2=50.0, L=90.0)
fig_cxa, anim_cx = animator_cx.animate_cavity(
    y0_initial=5.5,
    theta0_initial_deg=0.0,
    N_round_trips=30,
    ray_color="blue",
    save_animation=False,
)

### **13.6 Multiple-Ray Bundle in a Stable Cavity**

A bundle of five rays with launch angles from 0° to 2° in a symmetric concave–concave cavity ($R_1 = R_2 = -50$ cm, $L = 90$ cm, $g_1g_2 = 0.64$). The bundle spreads, refocuses, and stays confined — direct visual evidence of stability across a range of launch conditions. `interactive_prompt=False` keeps the cell reproducible.

In [ ]:
cavity_bundle = CavityRayTracing(R1=-50, R2=-50, L=90)
fig_mb, anim_mb = multiple_ray_trace_cavity(
    cavity_bundle,
    y0_initial=8,
    N_round_trips=10,
    colorscale="bwr",
    arc_angle=40,
    fps=40,
    interactive_prompt=False,
)

### **13.7 Stability Diagram with a Cavity Marker**

The $g_1$–$g_2$ diagram with the operating point of a symmetric cavity ($R_1 = R_2 = -80$ cm, $L = 100$ cm, so $g_1 = g_2 = -0.25$ and $g_1g_2 = 0.0625$): a stable point in the third-quadrant lobe of the stable region. Pass `save_figure=True` to export the diagram to `OUTPUTS/PLOTS`.

In [ ]:
cavity_sd = CavityRayTracing(R1=-80, R2=-80, L=100)
fig_sd, ax_sd = plot_stability_diagram(cavity_sd, save_figure=True)

### **13.8 Batch Animation Export (Optional)**

Generates and saves a GIF for every concave–convex case in `CAVITY_CASES`. This renders many animation frames and requires FFmpeg, so it is disabled by default — set `RUN_BATCH_ANIMATION_EXPORT = True` to run it. Outputs go to `OUTPUTS/ANIMATIONS`.

In [ ]:
# Batch-export animations for all concave-convex cases
RUN_BATCH_ANIMATION_EXPORT = True

if RUN_BATCH_ANIMATION_EXPORT:
    for case_name, params in CAVITY_CASES["concave_concave"].items():
        print(f"\nGenerating and saving animation for: {case_name}")

        animator = AnimateCavityRayTracing(
            R1=params["R1"], R2=params["R2"], L=params["L"]
        )

        print("Used Parameter Values")
        for param_key, param_val in params.items():
            print(f"{param_key:<20}{str(param_val):>10}")

        fig, anim = animator.animate_cavity(
            y0_initial=params["y0_initial"],
            theta0_initial_deg=params["theta0_initial_deg"],
            N_round_trips=params["N_round_trips"],
            arc_angle=params["arc_angle"],
            ray_color=params["ray_color"],
            save_animation=True,
            filename=None,
        )

### **13.9 Batch Image Export (Optional)**

In [ ]:
RUN_BATCH_IMAGE_EXPORT = False  # Set to True to enable batch image export

if RUN_BATCH_IMAGE_EXPORT:
    for case_name, params in CAVITY_CASES["concave_concave"].items():
        print(f"\nGenerating and saving static plot for: {case_name}")

        cavity = CavityRayTracing(R1=params["R1"], R2=params["R2"], L=params["L"])

        print("Used Parameter Values")
        for param_key, param_val in params.items():
            print(f"{param_key:<20}{str(param_val):>10}")

        fig, ax = cavity.visualize_single_ray(
            y0_initial=params["y0_initial"],
            theta0_initial_deg=params["theta0_initial_deg"],
            N_round_trips=params["N_round_trips"],
            arc_angle=params["arc_angle"],
            ray_color=params["ray_color"],
            save_figure=True,
            filename=None,
        )

<a id="14-references"></a>
## 14. References

1. A. E. Siegman, *Lasers*, University Science Books, 1986 — chapters on paraxial ray optics and resonator stability.
2. H. Kogelnik and T. Li, "Laser Beams and Resonators," *Applied Optics*, **5**(10), 1550–1567, 1966 — the classic reference for the $g$-parameter stability diagram.
3. B. E. A. Saleh and M. C. Teich, *Fundamentals of Photonics*, Wiley — chapters on ray optics and resonators.
4. [Introduction to Laser, NPTEL - IIT Delhi](https://youtu.be/Ab1nxxkgjH8?si=J2JF6P0DlmLRzfe9)